# 02 - Silver - Mapeamento e Unificação de Colunas (Glue Notebook)

**Tech Challenge - Fase 3 - Big Data to Analytics**

Objetivo: ler os 3 Parquets, aplicar o mapeamento de colunas (nomes diferentes por ano) e gerar uma **única base unificada e consistente**.

## 1. Sessão do Glue Notebook

In [5]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 60 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


In [1]:
import sys
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Sessao do Glue Notebook pronta.")

Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 60
Session ID: 69e11b8e-c779-494a-9944-87b644efab34
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 69e11b8e-c779-494a-9944-87b644efab34 to get into ready status...
Session 69e11b8e-c779-494a-9944-87b644efab34 has been created.
Sessao do Glue Notebook pronta.


## 2. Configuração

In [2]:
NOME_BUCKET = "tech-challenge-fase-3-grupo-94"
CAMINHO_SILVER = f"s3://{NOME_BUCKET}/silver/state_of_data/"
CAMINHO_SILVER_UNIFICADO = f"s3://{NOME_BUCKET}/silver/state_of_data_unificado/"

DATABASE = "workspace"
TABELA_UNIFICADA = "tb_state_of_data_silver"

## 3. Ler cada ano separadamente e adiciona o ano da pesquisa

In [3]:
df_2023 = spark.read.parquet(CAMINHO_SILVER+"/ano_pesquisa=2023/")
df_2024 = spark.read.parquet(CAMINHO_SILVER+"/ano_pesquisa=2024/")
df_2025 = spark.read.parquet(CAMINHO_SILVER+"/ano_pesquisa=2025/")

df_2023 = df_2023.withColumn("ano_pesquisa", lit(2023))
df_2024 = df_2024.withColumn("ano_pesquisa", lit(2024))
df_2025 = df_2025.withColumn("ano_pesquisa", lit(2025))

print("Colunas 2023:", len(df_2023.columns))
print("Colunas 2024:", len(df_2024.columns))
print("Colunas 2025:", len(df_2025.columns))

Colunas 2023: 400
Colunas 2024: 404
Colunas 2025: 389


## 4. Conferir os nomes reais das colunas antes de mapear


In [4]:
print("Exemplo de colunas - 2023:")
print(df_2023.columns[:15])

print("\nExemplo de colunas - 2024:")
print(df_2024.columns[:15])

print("\nExemplo de colunas - 2025:")
print(df_2025.columns[:15])

Exemplo de colunas - 2023:
["('P0', 'id')", "('P1_a ', 'Idade')", "('P1_a_1 ', 'Faixa idade')", "('P1_b ', 'Genero')", "('P1_c ', 'Cor/raca/etnia')", "('P1_d ', 'PCD')", "('P1_e ', 'experiencia_profissional_prejudicada')", "('P1_e_1 ', 'Não acredito que minha experiência profissional seja afetada')", "('P1_e_2 ', 'Experiencia prejudicada devido a minha Cor Raça Etnia')", "('P1_e_3 ', 'Experiencia prejudicada devido a minha identidade de gênero')", "('P1_e_4 ', 'Experiencia prejudicada devido ao fato de ser PCD')", "('P1_f ', 'aspectos_prejudicados')", "('P1_f_1', 'Quantidade de oportunidades de emprego/vagas recebidas')", "('P1_f_2', 'Senioridade das vagas recebidas em relação à sua experiência')", "('P1_f_3', 'Aprovação em processos seletivos/entrevistas')"]

Exemplo de colunas - 2024:
['0.a_token', '0.d_data/hora_envio', '1.a_idade', '1.a.1_faixa_idade', '1.b_genero', '1.c_cor/raca/etnia', '1.d_pcd', '1.e_experiencia_profissional_prejudicada', '1.e.1_Não acredito que minha experiênci

## 5. Dicionários de mapeamento (de-para)

Um dicionário por ano: `{nome_original: nome_padronizado}`.

In [ ]:
mapa_2023 = {
    "('P0', 'id')": "id",
    "('P1_a ', 'Idade')": "Idade",
    "('P1_a_1 ', 'Faixa idade')": "Faixa_idade",
    "('P1_b ', 'Genero')": "Genero",
    "('P1_c ', 'Cor/raca/etnia')": "Cor_raca_etnia",
    "('P1_d ', 'PCD')": "PCD",
    "('P1_e ', 'experiencia_profissional_prejudicada')": "experiencia_profissional_prejudicada",
    "('P1_e_1 ', 'Não acredito que minha experiência profissional seja afetada')": "Nao_acredito_que_minha_experiencia_profissional_seja_afetada",
    "('P1_e_2 ', 'Experiencia prejudicada devido a minha Cor Raça Etnia')": "Experiencia_prejudicada_devido_a_minha_Cor_Raca_Etnia",
    "('P1_e_3 ', 'Experiencia prejudicada devido a minha identidade de gênero')": "Experiencia_prejudicada_devido_a_minha_identidade_de_genero",
    "('P1_e_4 ', 'Experiencia prejudicada devido ao fato de ser PCD')": "Experiencia_prejudicada_devido_ao_fato_de_ser_PCD",
    "('P1_f ', 'aspectos_prejudicados')": "aspectos_prejudicados",
    "('P1_f_1', 'Quantidade de oportunidades de emprego/vagas recebidas')": "Quantidade_de_oportunidades_de_emprego_vagas_recebidas",
    "('P1_f_2', 'Senioridade das vagas recebidas em relação à sua experiência')": "Senioridade_das_vagas_recebidas_em_relacao_a_sua_experiencia",
    "('P1_f_3', 'Aprovação em processos seletivos/entrevistas')": "Aprovacao_em_processos_seletivos_entrevistas",
    "('P1_f_4', 'Oportunidades de progressão de carreira')": "Oportunidades_de_progressao_de_carreira",
    "('P1_f_5', 'Velocidade de progressão de carreira')": "Velocidade_de_progressao_de_carreira",
    "('P1_f_6', 'Nível de cobrança no trabalho/Stress no trabalho')": "Nivel_de_cobranca_no_trabalho_Stress_no_trabalho",
    "('P1_f_7', 'Atenção dada diante das minhas opiniões e ideias')": "Atencao_dada_diante_das_minhas_opinioes_e_ideias",
    "('P1_f_8', 'Relação com outros membros da empresa, em momentos de trabalho')": "Relacao_com_outros_membros_da_empresa_em_momentos_de_trabalho",
    "('P1_f_9', 'Relação com outros membros da empresa, em momentos de integração e outros momentos fora do trabalho')": "Relacao_com_outros_membros_da_empresa_em_momentos_de_integracao_e_outros_momentos_fora_do_trabalho",
    "('P1_g ', 'vive_no_brasil')": "vive_no_brasil",
    "('P1_i ', 'Estado onde mora')": "Estado_onde_mora",
    "('P1_i_1 ', 'uf onde mora')": "uf_onde_mora",
    "('P1_i_2 ', 'Regiao onde mora')": "Regiao_onde_mora",
    "('P1_k ', 'Regiao de origem')": "Regiao_de_origem",
    "('P1_j ', 'Mudou de Estado?')": "Mudou_de_Estado",
    "('P1_l ', 'Nivel de Ensino')": "Nivel_de_Ensino",
    "('P1_m ', 'Área de Formação')": "Area_de_Formacao",
    "('P2_a ', 'Qual sua situação atual de trabalho?')": "Qual_sua_situacao_atual_de_trabalho",
    "('P2_b ', 'Setor')": "Setor",
    "('P2_c ', 'Numero de Funcionarios')": "Numero_de_Funcionarios",
    "('P2_d ', 'Gestor?')": "Gestor",
    "('P2_e ', 'Cargo como Gestor')": "Cargo_como_Gestor",
    "('P2_f ', 'Cargo Atual')": "Cargo_Atual",
    "('P2_g ', 'Nivel')": "Nivel",
    "('P2_h ', 'Faixa salarial')": "Faixa_salarial",
    "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')": "Quanto_tempo_de_experiencia_na_area_de_dados_voce_tem",
    "('P2_j ', 'Quanto tempo de experiência na área de TI/Engenharia de Software você teve antes de começar a trabalhar na área de dados?')": "Quanto_tempo_de_experiencia_na_area_de_TI_Engenharia_de_Software_voce_teve_antes_de_comecar_a_trabalhar_na_area_de_dados",
    "('P2_k ', 'Você está satisfeito na sua empresa atual?')": "Voce_esta_satisfeito_na_sua_empresa_atual",
    "('P2_l ', 'Qual o principal motivo da sua insatisfação com a empresa atual?')": "Qual_o_principal_motivo_da_sua_insatisfacao_com_a_empresa_atual",
    "('P2_l_2 ', 'Salário atual não corresponde ao mercado')": "Salario_atual_nao_corresponde_ao_mercado",
    "('P2_l_5 ', 'Gostaria de receber mais benefícios')": "Gostaria_de_receber_mais_beneficios",
    "('P2_l_6 ', 'O clima de trabalho/ambiente não é bom')": "O_clima_de_trabalho_ambiente_nao_e_bom",
    "('P2_l_1 ', 'Falta de oportunidade de crescimento no emprego atual')": "Falta_de_oportunidade_de_crescimento_no_emprego_atual",
    "('P2_l_7 ', 'Falta de maturidade analítica na empresa')": "Falta_de_maturidade_analitica_na_empresa",
    "('P2_l_3 ', 'Não tenho uma boa relação com meu líder/gestor')": "Nao_tenho_uma_boa_relacao_com_meu_lider_gestor",
    "('P2_l_4 ', 'Gostaria de trabalhar em em outra área de atuação')": "Gostaria_de_trabalhar_em_em_outra_area_de_atuacao",
    "('P2_m ', 'Você participou de entrevistas de emprego nos últimos 6 meses?')": "Voce_participou_de_entrevistas_de_emprego_nos_ultimos_6_meses",
    "('P2_n ', 'Você pretende mudar de emprego nos próximos 6 meses?')": "Voce_pretende_mudar_de_emprego_nos_proximos_6_meses",
    "('P2_o ', 'Quais os principais critérios que você leva em consideração no momento de decidir onde trabalhar?')": "Quais_os_principais_criterios_que_voce_leva_em_consideracao_no_momento_de_decidir_onde_trabalhar",
    "('P2_o_1 ', 'Remuneração/Salário')": "Remuneracao_Salario",
    "('P2_o_2 ', 'Benefícios')": "Beneficios",
    "('P2_o_3 ', 'Propósito do trabalho e da empresa')": "Proposito_do_trabalho_e_da_empresa",
    "('P2_o_4 ', 'Flexibilidade de trabalho remoto')": "Flexibilidade_de_trabalho_remoto",
    "('P2_o_5 ', 'Ambiente e clima de trabalho')": "Ambiente_e_clima_de_trabalho",
    "('P2_o_6 ', 'Oportunidade de aprendizado e trabalhar com referências na área')": "Oportunidade_de_aprendizado_e_trabalhar_com_referencias_na_area",
    "('P2_o_7 ', 'Plano de carreira e oportunidades de crescimento profissional')": "Plano_de_carreira_e_oportunidades_de_crescimento_profissional",
    "('P2_o_8 ', 'Maturidade da empresa em termos de tecnologia e dados')": "Maturidade_da_empresa_em_termos_de_tecnologia_e_dados",
    "('P2_o_9 ', 'Qualidade dos gestores e líderes')": "Qualidade_dos_gestores_e_lideres",
    "('P2_o_10 ', 'Reputação que a empresa tem no mercado')": "Reputacao_que_a_empresa_tem_no_mercado",
    "('P2_q ', 'Empresa que trabaha passou por layoff em 2023')": "Empresa_que_trabaha_passou_por_layoff_em_2023",
    "('P2_r ', 'Atualmente qual a sua forma de trabalho?')": "Atualmente_qual_a_sua_forma_de_trabalho",
    "('P2_s ', 'Qual a forma de trabalho ideal para você?')": "Qual_a_forma_de_trabalho_ideal_para_voce",
    "('P2_t ', 'Caso sua empresa decida pelo modelo 100% presencial qual será sua atitude?')": "Caso_sua_empresa_decida_pelo_modelo_100_presencial_qual_sera_sua_atitude",
    "('P3_a ', 'Qual o número aproximado de pessoas que atuam com dados na sua empresa hoje?')": "Qual_o_numero_aproximado_de_pessoas_que_atuam_com_dados_na_sua_empresa_hoje",
    "('P3_b ', 'Quais desses papéis/cargos fazem parte do time (ou chapter) de dados da sua empresa?')": "Quais_desses_papeis_cargos_fazem_parte_do_time_ou_chapter_de_dados_da_sua_empresa",
    "('P3_b_1 ', 'Analytics Engineer')": "Analytics_Engineer",
    "('P3_b_2 ', 'Engenharia de Dados/Data Engineer')": "Engenharia_de_Dados_Data_Engineer",
    "('P3_b_3 ', 'Analista de Dados/Data Analyst')": "Analista_de_Dados_Data_Analyst",
    "('P3_b_4 ', 'Cientista de Dados/Data Scientist')": "Cientista_de_Dados_Data_Scientist",
    "('P3_b_5 ', 'Database Administrator/DBA')": "Database_Administrator_DBA",
    "('P3_b_6 ', 'Analista de Business Intelligence/BI')": "Analista_de_Business_Intelligence_BI",
    "('P3_b_7 ', 'Arquiteto de Dados/Data Architect')": "Arquiteto_de_Dados_Data_Architect",
    "('P3_b_8 ', 'Data Product Manager/DPM')": "Data_Product_Manager_DPM",
    "('P3_b_9 ', 'Business Analyst')": "Business_Analyst",
    "('P3_c ', 'Quais dessas responsabilidades fazem parte da sua rotina atual de trabalho como gestor?')": "Quais_dessas_responsabilidades_fazem_parte_da_sua_rotina_atual_de_trabalho_como_gestor",
    "('P3_c_1 ', 'Pensar na visão de longo prazo de dados da empresa e fortalecimento da cultura analítica da companhia.')": "Pensar_na_visao_de_longo_prazo_de_dados_da_empresa_e_fortalecimento_da_cultura_analitica_da_companhia",
    "('P3_c_2 ', 'Organização de treinamentos e iniciativas com o objetivo de aumentar a maturidade analítica das áreas de negócios.')": "Organizacao_de_treinamentos_e_iniciativas_com_o_objetivo_de_aumentar_a_maturidade_analitica_das_areas_de_negocios",
    "('P3_c_3 ', 'Atração, seleção e contratação de talentos para o time de dados.')": "Atracao_selecao_e_contratacao_de_talentos_para_o_time_de_dados",
    "('P3_c_4 ', 'Decisão sobre contratação de ferramentas e tecnologias relacionadas a dados.')": "Decisao_sobre_contratacao_de_ferramentas_e_tecnologias_relacionadas_a_dados",
    "('P3_c_5 ', 'Sou gestor da equipe responsável pela engenharia de dados e por manter o Data Lake da empresa como fonte única dos dados, garantindo a qualidade e confiabilidade da informação.')": "Sou_gestor_da_equipe_responsavel_pela_engenharia_de_dados_e_por_manter_o_Data_Lake_da_empresa_como_fonte_unica_dos_dados_garantindo_a_qualidade_e_confiabilidade_da_informacao",
    "('P3_c_6 ', 'Sou gestor da equipe responsável pela entrega de dados, estudos, relatórios e dashboards para as áreas de negócio da empresa.')": "Sou_gestor_da_equipe_responsavel_pela_entrega_de_dados_estudos_relatorios_e_dashboards_para_as_areas_de_negocio_da_empresa",
    "('P3_c_7 ', 'Sou gestor da equipe responsável por iniciativas e projetos envolvendo Inteligência Artificial e Machine Learning.')": "Sou_gestor_da_equipe_responsavel_por_iniciativas_e_projetos_envolvendo_Inteligencia_Artificial_e_Machine_Learning",
    "('P3_c_8 ', 'Apesar de ser gestor ainda atuo na parte técnica, construindo soluções/análises/modelos etc.')": "Apesar_de_ser_gestor_ainda_atuo_na_parte_tecnica_construindo_solucoes_analises_modelos_etc",
    "('P3_c_9 ', 'Gestão de projetos de dados, cuidando das etapas, equipes envolvidas, atingimento dos objetivos etc.')": "Gestao_de_projetos_de_dados_cuidando_das_etapas_equipes_envolvidas_atingimento_dos_objetivos_etc",
    "('P3_c_10 ', 'Gestão de produtos de dados, cuidando da visão dos produtos, backlog, feedback de usuários etc.')": "Gestao_de_produtos_de_dados_cuidando_da_visao_dos_produtos_backlog_feedback_de_usuarios_etc",
    "('P3_c_11 ', 'Gestão de pessoas, apoio no desenvolvimento das pessoas, evolução de carreira')": "Gestao_de_pessoas_apoio_no_desenvolvimento_das_pessoas_evolucao_de_carreira",
    "('P3_d ', 'Quais são os 3 maiores desafios que você tem como gestor no atual momento?')": "Quais_sao_os_3_maiores_desafios_que_voce_tem_como_gestor_no_atual_momento",
    "('P3_d_1 ', 'a Contratar novos talentos.')": "a_Contratar_novos_talentos",
    "('P3_d_2 ', 'b Reter talentos.')": "b_Reter_talentos",
    "('P3_d_3 ', 'c Convencer a empresa a aumentar os investimentos na área de dados.')": "c_Convencer_a_empresa_a_aumentar_os_investimentos_na_area_de_dados",
    "('P3_d_4 ', 'd Gestão de equipes no ambiente remoto.')": "d_Gestao_de_equipes_no_ambiente_remoto",
    "('P3_d_5 ', 'e Gestão de projetos envolvendo áreas multidisciplinares da empresa.')": "e_Gestao_de_projetos_envolvendo_areas_multidisciplinares_da_empresa",
    "('P3_d_6 ', 'f Organizar as informações e garantir a qualidade e confiabilidade.')": "f_Organizar_as_informacoes_e_garantir_a_qualidade_e_confiabilidade",
    "('P3_d_7 ', 'g Conseguir processar e armazenar um alto volume de dados.')": "g_Conseguir_processar_e_armazenar_um_alto_volume_de_dados",
    "('P3_d_8 ', 'h Conseguir gerar valor para as áreas de negócios através de estudos e experimentos.')": "h_Conseguir_gerar_valor_para_as_areas_de_negocios_atraves_de_estudos_e_experimentos",
    "('P3_d_9 ', 'i Desenvolver e manter modelos Machine Learning em produção.')": "i_Desenvolver_e_manter_modelos_Machine_Learning_em_producao",
    "('P3_d_10 ', 'j Gerenciar a expectativa das áreas de negócio em relação as entregas das equipes de dados.')": "j_Gerenciar_a_expectativa_das_areas_de_negocio_em_relacao_as_entregas_das_equipes_de_dados",
    "('P3_d_11 ', 'k Garantir a manutenção dos projetos e modelos em produção, em meio ao crescimento da empresa.')": "k_Garantir_a_manutencao_dos_projetos_e_modelos_em_producao_em_meio_ao_crescimento_da_empresa",
    "('P3_d_12 ', 'Conseguir levar inovação para a empresa através dos dados.')": "Conseguir_levar_inovacao_para_a_empresa_atraves_dos_dados",
    "('P3_d_13 ', 'Garantir retorno do investimento (ROI) em projetos de dados.')": "Garantir_retorno_do_investimento_ROI_em_projetos_de_dados",
    "('P3_d_14 ', 'Dividir o tempo entre entregas técnicas e gestão.')": "Dividir_o_tempo_entre_entregas_tecnicas_e_gestao",
    "('P3_e ', 'AI Generativa é uma prioridade em sua empresa?')": "AI_Generativa_e_uma_prioridade_em_sua_empresa",
    "('P3_f ', 'Tipos de uso de AI Generativa e LLMs na empresa')": "Tipos_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "('P3_f_1 ', 'Colaboradores usando AI generativa de forma independente e descentralizada')": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_P3f",
    "('P4_l_1 ', 'Colaboradores usando AI generativa de forma independente e descentralizada')": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_P4l",
    "('P3_f_2 ', 'Direcionamento centralizado do uso de AI generativa')": "Direcionamento_centralizado_do_uso_de_AI_generativa_P3f",
    "('P4_l_2 ', 'Direcionamento centralizado do uso de AI generativa')": "Direcionamento_centralizado_do_uso_de_AI_generativa_P4l",
    "('P3_f_3 ', 'Desenvolvedores utilizando Copilots')": "Desenvolvedores_utilizando_Copilots_P3f",
    "('P4_l_3 ', 'Desenvolvedores utilizando Copilots')": "Desenvolvedores_utilizando_Copilots_P4l",
    "('P3_f_4 ', 'AI Generativa e LLMs para melhorar produtos externos')": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos",
    "('P3_f_5 ', 'AI Generativa e LLMs para melhorar produtos internos para os colaboradores')": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_P3f",
    "('P4_l_5 ', 'AI Generativa e LLMs para melhorar produtos internos para os colaboradores')": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_P4l",
    "('P3_f_6 ', 'IA Generativa e LLMs como principal frente do negócio')": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_P3f",
    "('P4_l_6 ', 'IA Generativa e LLMs como principal frente do negócio')": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_P4l",
    "('P3_f_7 ', 'IA Generativa e LLMs não é prioridade')": "IA_Generativa_e_LLMs_nao_e_prioridade_P3f",
    "('P4_l_7 ', 'IA Generativa e LLMs não é prioridade')": "IA_Generativa_e_LLMs_nao_e_prioridade_P4l",
    "('P3_f_8 ', 'Não sei opinar sobre o uso de IA Generativa e LLMs na empresa')": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_P3f",
    "('P4_l_8 ', 'Não sei opinar sobre o uso de IA Generativa e LLMs na empresa')": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_P4l",
    "('P3_g ', 'Motivos que levam a empresa a não usar AI Genrativa e LLMs')": "Motivos_que_levam_a_empresa_a_nao_usar_AI_Genrativa_e_LLMs",
    "('P3_g_1 ', 'Falta de compreensão dos casos de uso')": "Falta_de_compreensao_dos_casos_de_uso",
    "('P3_g_2 ', 'Falta de confiabilidade das saídas (alucinação dos modelos)')": "Falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos",
    "('P3_g_3 ', 'Incerteza em relação a regulamentação')": "Incerteza_em_relacao_a_regulamentacao",
    "('P3_g_4 ', 'Preocupações com segurança e privacidade de dados')": "Preocupacoes_com_seguranca_e_privacidade_de_dados",
    "('P3_g_5 ', 'Retorno sobre investimento (ROI) não comprovado de IA Generativa')": "Retorno_sobre_investimento_ROI_nao_comprovado_de_IA_Generativa",
    "('P3_g_6 ', 'Dados da empresa não estão prontos para uso de IA Generativa')": "Dados_da_empresa_nao_estao_prontos_para_uso_de_IA_Generativa",
    "('P3_g_7 ', 'Falta de expertise ou falta de recursos')": "Falta_de_expertise_ou_falta_de_recursos",
    "('P3_g_8 ', 'Alta direção da empresa não vê valor ou não vê como prioridade')": "Alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade",
    "('P3_g_9 ', 'Preocupações com propriedade intelectual')": "Preocupacoes_com_propriedade_intelectual",
    "('P4_a ', 'Mesmo que esse não seja seu cargo formal, você considera que sua atuação no dia a dia, reflete alguma das opções listadas abaixo?')": "Mesmo_que_esse_nao_seja_seu_cargo_formal_voce_considera_que_sua_atuacao_no_dia_a_dia_reflete_alguma_das_opcoes_listadas_abaixo",
    "('P4_a_1 ', 'Atuacao')": "Atuacao",
    "('P4_b ', 'Quais das fontes de dados listadas você já analisou ou processou no trabalho?')": "Quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho",
    "('P4_b_1 ', 'Dados relacionais (estruturados em bancos SQL)')": "Dados_relacionais_estruturados_em_bancos_SQL_P4b",
    "('P4_c_1 ', 'Dados relacionais (estruturados em bancos SQL)')": "Dados_relacionais_estruturados_em_bancos_SQL_P4c",
    "('P4_b_2 ', 'Dados armazenados em bancos NoSQL')": "Dados_armazenados_em_bancos_NoSQL_P4b",
    "('P4_c_2 ', 'Dados armazenados em bancos NoSQL')": "Dados_armazenados_em_bancos_NoSQL_P4c",
    "('P4_b_3 ', 'Imagens')": "Imagens_P4b",
    "('P4_c_3 ', 'Imagens')": "Imagens_P4c",
    "('P4_b_4 ', 'Textos/Documentos')": "Textos_Documentos_P4b",
    "('P4_c_4 ', 'Textos/Documentos')": "Textos_Documentos_P4c",
    "('P4_b_5 ', 'Vídeos')": "Videos_P4b",
    "('P4_c_5 ', 'Vídeos')": "Videos_P4c",
    "('P4_b_6 ', 'Áudios')": "Audios_P4b",
    "('P4_c_6 ', 'Áudios')": "Audios_P4c",
    "('P4_b_7 ', 'Planilhas')": "Planilhas_P4b",
    "('P4_c_7 ', 'Planilhas')": "Planilhas_P4c",
    "('P4_b_8 ', 'Dados georeferenciados')": "Dados_georeferenciados_P4b",
    "('P4_c_8 ', 'Dados georeferenciados')": "Dados_georeferenciados_P4c",
    "('P4_c ', 'Entre as fontes de dados listadas, quais você utiliza na maior parte do tempo?')": "Entre_as_fontes_de_dados_listadas_quais_voce_utiliza_na_maior_parte_do_tempo",
    "('P4_d ', 'Quais das linguagens listadas abaixo você utiliza no trabalho?')": "Quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho_Linguagem",
    "('P4_d_1 ', 'SQL')": "SQL_Linguagem",
    "('P4_d_2 ', 'R ')": "R_Linguagem",
    "('P4_d_3 ', 'Python')": "Python_Linguagem",
    "('P4_d_4 ', 'C/C++/C#')": "C_C_C_Linguagem",
    "('P4_d_5 ', '.NET')": "NET_Linguagem",
    "('P4_d_6 ', 'Java')": "Java_Linguagem",
    "('P4_d_7 ', 'Julia')": "Julia_Linguagem",
    "('P4_d_8 ', 'SAS/Stata')": "SAS_Stata_Linguagem",
    "('P4_d_9 ', 'Visual Basic/VBA')": "Visual_Basic_VBA_Linguagem",
    "('P4_d_10 ', 'Scala')": "Scala_Linguagem",
    "('P4_d_11 ', 'Matlab')": "Matlab_Linguagem",
    "('P4_d_12 ', 'Rust')": "Rust_Linguagem",
    "('P4_d_13 ', 'PHP')": "PHP_Linguagem",
    "('P4_d_14 ', 'JavaScript')": "JavaScript_Linguagem",
    "('P4_d_15 ', 'Não utilizo nenhuma linguagem')": "Nao_utilizo_nenhuma_linguagem_Linguagem",
    "('P4_e ', 'Entre as linguagens listadas abaixo, qual é a que você mais utiliza no trabalho?')": "Entre_as_linguagens_listadas_abaixo_qual_e_a_que_voce_mais_utiliza_no_trabalho",
    "('P4_f ', 'Entre as linguagens listadas abaixo, qual é a sua preferida?')": "Entre_as_linguagens_listadas_abaixo_qual_e_a_sua_preferida",
    "('P4_g ', 'Quais dos bancos de dados/fontes de dados listados abaixo você utiliza no trabalho?')": "Quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho_BD",
    "('P4_g_1 ', 'MySQL')": "MySQL_BD",
    "('P4_g_2 ', 'Oracle')": "Oracle_BD",
    "('P4_g_3 ', 'SQL SERVER')": "SQL_SERVER_BD",
    "('P4_g_4 ', 'Amazon Aurora ou RDS')": "Amazon_Aurora_ou_RDS_BD",
    "('P4_g_5 ', 'DynamoDB')": "DynamoDB_BD",
    "('P4_g_6 ', 'CoachDB')": "CoachDB_BD",
    "('P4_g_7 ', 'Cassandra')": "Cassandra_BD",
    "('P4_g_8 ', 'MongoDB')": "MongoDB_BD",
    "('P4_g_9 ', 'MariaDB')": "MariaDB_BD",
    "('P4_g_10 ', 'Datomic')": "Datomic_BD",
    "('P4_g_11 ', 'S3')": "S3_BD",
    "('P4_g_12 ', 'PostgreSQL')": "PostgreSQL_BD",
    "('P4_g_13 ', 'ElasticSearch')": "ElasticSearch_BD",
    "('P4_g_14 ', 'DB2')": "DB2_BD",
    "('P4_g_15 ', 'Microsoft Access')": "Microsoft_Access_BD",
    "('P4_g_16 ', 'SQLite')": "SQLite_BD",
    "('P4_g_17 ', 'Sybase')": "Sybase_BD",
    "('P4_g_18 ', 'Firebase')": "Firebase_BD",
    "('P4_g_19 ', 'Vertica')": "Vertica_BD",
    "('P4_g_20 ', 'Redis')": "Redis_BD",
    "('P4_g_21 ', 'Neo4J')": "Neo4J_BD",
    "('P4_g_22 ', 'Google BigQuery')": "Google_BigQuery_BD",
    "('P4_g_23 ', 'Google Firestore')": "Google_Firestore_BD",
    "('P4_g_24 ', 'Amazon Redshift')": "Amazon_Redshift_BD",
    "('P4_g_25 ', 'Amazon Athena')": "Amazon_Athena_BD",
    "('P4_g_26 ', 'Snowflake')": "Snowflake_BD",
    "('P4_g_27 ', 'Databricks')": "Databricks_BD",
    "('P6_b_20 ', 'Databricks')": "Databricks_P6b",
    "('P7_b_20 ', 'Databricks')": "Databricks_ETL",
    "('P4_g_28 ', 'HBase')": "HBase_BD",
    "('P4_g_29 ', 'Presto')": "Presto_BD",
    "('P4_g_30 ', 'Splunk')": "Splunk_BD",
    "('P4_g_31 ', 'SAP HANA')": "SAP_HANA_BD",
    "('P4_g_32 ', 'Hive')": "Hive_BD",
    "('P4_g_33 ', 'Firebird')": "Firebird_BD",
    "('P4_h ', 'Dentre as opções listadas, qual sua Cloud preferida?')": "Dentre_as_opcoes_listadas_qual_sua_Cloud_preferida_cloud",
    "('P4_h_1 ', 'Azure (Microsoft)')": "Azure_Microsoft_cloud",
    "('P4_h_2 ', 'Amazon Web Services (AWS)')": "Amazon_Web_Services_AWS_cloud",
    "('P4_h_3 ', 'Google Cloud (GCP)')": "Google_Cloud_GCP_cloud",
    "('P4_h_4 ', 'Oracle Cloud')": "Oracle_Cloud",
    "('P4_h_5 ', 'IBM')": "IBM_cloud",
    "('P4_h_6 ', 'Servidores On Premise/Não utilizamos Cloud')": "Servidores_On_Premise_Nao_utilizamos_Cloud_cloud",
    "('P4_h_7 ', 'Cloud Própria')": "Cloud_Propria_cloud",
    "('P4_i ', 'Cloud preferida')": "Cloud_preferida",
    "('P4_j ', 'Ferramenta de BI utilizada no dia a dia')": "Ferramenta_de_BI_utilizada_no_dia_a_dia_BI",
    "('P4_j_1 ', 'Microsoft PowerBI')": "Microsoft_PowerBI_BI",
    "('P4_j_2 ', 'Qlik View/Qlik Sense')": "Qlik_View_Qlik_Sense_BI",
    "('P4_j_3 ', 'Tableau')": "Tableau_BI",
    "('P4_j_4 ', 'Metabase')": "Metabase_BI",
    "('P4_j_5 ', 'Superset')": "Superset_BI",
    "('P4_j_6 ', 'Redash')": "Redash_BI",
    "('P4_j_7 ', 'Looker')": "Looker_BI",
    "('P4_j_8 ', 'Looker Studio(Google Data Studio)')": "Looker_Studio_Google_Data_Studio_BI",
    "('P4_j_9 ', 'Amazon Quicksight')": "Amazon_Quicksight_BI",
    "('P4_j_10 ', 'Mode')": "Mode_BI",
    "('P4_j_11 ', 'Alteryx')": "Alteryx_BI",
    "('P6_b_9 ', 'Alteryx')": "Alteryx_P6b",
    "('P7_b_9 ', 'Alteryx')": "Alteryx_ETL",
    "('P4_j_12 ', 'MicroStrategy')": "MicroStrategy_BI",
    "('P4_j_13 ', 'IBM Analytics/Cognos')": "IBM_Analytics_Cognos_BI",
    "('P4_j_14 ', 'SAP Business Objects/SAP Analytics')": "SAP_Business_Objects_SAP_Analytics_BI",
    "('P4_j_15 ', 'Oracle Business Intelligence')": "Oracle_Business_Intelligence_BI",
    "('P4_j_16 ', 'Salesforce/Einstein Analytics')": "Salesforce_Einstein_Analytics_BI",
    "('P4_j_17 ', 'Birst')": "Birst_BI",
    "('P4_j_18 ', 'SAS Visual Analytics')": "SAS_Visual_Analytics_BI",
    "('P4_j_19 ', 'Grafana')": "Grafana_BI",
    "('P4_j_20 ', 'TIBCO Spotfire')": "TIBCO_Spotfire_BI",
    "('P4_j_21 ', 'Pentaho')": "Pentaho_BI",
    "('P6_b_8 ', 'Pentaho')": "Pentaho_P6b",
    "('P7_b_8 ', 'Pentaho')": "Pentaho_ETL",
    "('P4_j_22 ', 'Fazemos todas as análises utilizando apenas Excel ou planilhas do google')": "Fazemos_todas_as_analises_utilizando_apenas_Excel_ou_planilhas_do_google_BI",
    "('P4_j_23 ', 'Não utilizo nenhuma ferramenta de BI no trabalho')": "Nao_utilizo_nenhuma_ferramenta_de_BI_no_trabalho_BI",
    "('P4_k ', 'Qual sua ferramenta de BI preferida?')": "Qual_sua_ferramenta_de_BI_preferida",
    "('P4_l ', 'Qual o tipo de uso de AI Generativa e LLMs na empresa')": "Qual_o_tipo_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "('P4_l_4 ', 'AI Generativa e LLMs para melhorar produtos externos para os clientes finais')": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos_para_os_clientes_finais",
    "('P4_m ', 'Utiliza ChatGPT ou LLMs no trabalho?')": "Utiliza_ChatGPT_ou_LLMs_no_trabalho",
    "('P4_m_1 ', 'Não uso soluções de AI Generativa com foco em produtividade')": "Nao_uso_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "('P4_m_2 ', 'Uso soluções gratuitas de AI Generativa com foco em produtividade')": "Uso_solucoes_gratuitas_de_AI_Generativa_com_foco_em_produtividade",
    "('P4_m_3 ', 'Uso e pago pelas soluções de AI Generativa com foco em produtividade')": "Uso_e_pago_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "('P4_m_4 ', 'A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade')": "A_empresa_que_trabalho_paga_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "('P4_m_5 ', 'Uso soluções do tipo Copilot')": "Uso_solucoes_do_tipo_Copilot",
    "('P5_a ', 'Qual seu objetivo na área de dados?')": "Qual_seu_objetivo_na_area_de_dados",
    "('P5_b ', 'Qual oportunidade você está buscando?')": "Qual_oportunidade_voce_esta_buscando",
    "('P5_c ', 'Há quanto tempo você busca uma oportunidade na área de dados?')": "Ha_quanto_tempo_voce_busca_uma_oportunidade_na_area_de_dados",
    "('P5_d ', 'Como tem sido a busca por um emprego na área de dados?')": "Como_tem_sido_a_busca_por_um_emprego_na_area_de_dados",
    "('P6_a ', 'Quais das opções abaixo fazem parte da sua rotina no trabalho atual como engenheiro de dados?')": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_como_engenheiro_de_dados",
    "('P6_a_1 ', 'Desenvolvo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.')": "Desenvolvo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "('P6_a_2 ', 'Realizo construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.')": "Realizo_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "('P6_a_3 ', 'Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.')": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_P6a",
    "('P7_a_3 ', 'Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.')": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_P7a",
    "('P6_a_4 ', 'Atuo na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.')": "Atuo_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "('P6_a_5 ', 'Modelo soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.')": "Modelo_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "('P6_a_6 ', 'Desenvolvo/cuido da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.')": "Desenvolvo_cuido_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "('P6_a_7 ', 'Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.')": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "('P6_a_8 ', 'Cuido da qualidade dos dados, metadados e dicionário de dados.')": "Cuido_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "('P6_a_9 ', 'Nenhuma das opções listadas refletem meu dia a dia.')": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_P6a",
    "('P6_h_9 ', 'Nenhuma das opções listadas refletem meu dia a dia.')": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_P6h",
    "('P7_a_10 ', 'Nenhuma das opções listadas refletem meu dia a dia.')": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_P7a",
    "('P7_d_10 ', 'Nenhuma das opções listadas refletem meu dia a dia.')": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_P7d",
    "('P6_b ', 'Quais as ferramentas/tecnologias de ETL que você utiliza no trabalho como Data Engineer?')": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Engineer",
    "('P6_b_1 ', 'Scripts Python')": "Scripts_Python_P6b",
    "('P7_b_1 ', 'Scripts Python')": "Scripts_Python_ETL",
    "('P6_b_2 ', 'SQL & Stored Procedures')": "SQL_Stored_Procedures_P6b",
    "('P7_b_2 ', 'SQL & Stored Procedures')": "SQL_Stored_Procedures_ETL",
    "('P6_b_3 ', 'Apache Airflow')": "Apache_Airflow_P6b",
    "('P7_b_3 ', 'Apache Airflow')": "Apache_Airflow_ETL",
    "('P6_b_4 ', 'Apache NiFi')": "Apache_NiFi_P6b",
    "('P7_b_4 ', 'Apache NiFi')": "Apache_NiFi_ETL",
    "('P6_b_5 ', 'Luigi')": "Luigi_P6b",
    "('P7_b_5 ', 'Luigi')": "Luigi_ETL",
    "('P6_b_6 ', 'AWS Glue')": "AWS_Glue_P6b",
    "('P7_b_6 ', 'AWS Glue')": "AWS_Glue_ETL",
    "('P6_b_7 ', 'Talend')": "Talend_P6b",
    "('P7_b_7 ', 'Talend')": "Talend_ETL",
    "('P6_b_10 ', 'Stitch')": "Stitch_P6b",
    "('P7_b_10 ', 'Stitch')": "Stitch_ETL",
    "('P6_b_11 ', 'Fivetran')": "Fivetran_P6b",
    "('P7_b_11 ', 'Fivetran')": "Fivetran_ETL",
    "('P6_b_12 ', 'Google Dataflow')": "Google_Dataflow_P6b",
    "('P7_b_12 ', 'Google Dataflow')": "Google_Dataflow_ETL",
    "('P6_b_13 ', 'Oracle Data Integrator')": "Oracle_Data_Integrator_P6b",
    "('P7_b_13 ', 'Oracle Data Integrator')": "Oracle_Data_Integrator_ETL",
    "('P6_b_14 ', 'IBM DataStage')": "IBM_DataStage_P6b",
    "('P7_b_14 ', 'IBM DataStage')": "IBM_DataStage_ETL",
    "('P6_b_15 ', 'SAP BW ETL')": "SAP_BW_ETL_P6b",
    "('P7_b_15 ', 'SAP BW ETL')": "SAP_BW_ETL_ETL",
    "('P6_b_16 ', 'SQL Server Integration Services (SSIS))": "SQL_Server_Integration_Services_SSIS_P6b",
    "('P7_b_16 ', 'SQL Server Integration Services (SSIS)')": "SQL_Server_Integration_Services_SSIS_ETL",
    "('P6_b_17 ', 'SAS Data Integration')": "SAS_Data_Integration_P6b",
    "('P7_b_17 ', 'SAS Data Integration')": "SAS_Data_Integration_ETL",
    "('P6_b_18 ', 'Qlik Sense')": "Qlik_Sense_P6b",
    "('P7_b_18 ', 'Qlik Sense')": "Qlik_Sense_ETL",
    "('P6_b_19 ', 'Knime')": "Knime_P6b",
    "('P7_b_19 ', 'Knime')": "Knime_ETL",
    "('P6_b_21 ', 'Não utilizo ferramentas de ETL')": "Nao_utilizo_ferramentas_de_ETL_P6b",
    "('P7_b_21 ', 'Não utilizo ferramentas de ETL')": "Nao_utilizo_ferramentas_de_ETL_ETL",
    "('P6_c ', 'Sua organização possui um Data Lake?')": "Sua_organizacao_possui_um_Data_Lake",
    "('P6_d ', 'Qual tecnologia utilizada como plataforma do Data Lake?')": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Lake",
    "('P6_e ', 'Sua organização possui um Data Warehouse?')": "Sua_organizacao_possui_um_Data_Warehouse",
    "('P6_f ', 'Qual tecnologia utilizada como plataforma do Data Warehouse?')": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Warehouse",
    "('P6_g ', 'Quais as ferramentas de gestão de Qualidade de dados, Metadados e catálogo de dados você utiliza no trabalho?')": "Quais_as_ferramentas_de_gestao_de_Qualidade_de_dados_Metadados_e_catalogo_de_dados_voce_utiliza_no_trabalho",
    "('P6_h ', 'Em qual das opções abaixo você gasta a maior parte do seu tempo?')": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo",
    "('P6_h_1 ', 'Desenvolvendo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.')": "Desenvolvendo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "('P6_h_2 ', 'Realizando construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.')": "Realizando_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "('P6_h_3 ', 'Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.')": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_P6h",
    "('P7_d_3 ', 'Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.')": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_P7d",
    "('P6_h_4 ', 'Atuando na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.')": "Atuando_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "('P6_h_5 ', 'Modelando soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.')": "Modelando_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "('P6_h_6 ', 'Desenvolvendo/cuidando da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.')": "Desenvolvendo_cuidando_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "('P6_h_7 ', 'Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.')": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "('P6_h_8 ', 'Cuidando da qualidade dos dados, metadados e dicionário de dados.')": "Cuidando_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "('P7_1 ', 'Quais das opções abaixo fazem parte da sua rotina no trabalho atual com análise de dados?')": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_analise_de_dados",
    "('P7_a_1 ', 'Processo e analiso dados utilizando linguagens de programação como Python, R etc.')": "Processo_e_analiso_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "('P7_a_2 ', 'Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.')": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_P7a",
    "('P8_a_7 ', 'Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc')": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_P8a",
    "('P7_a_4 ', 'Utilizo API's para extrair dados e complementar minhas análises.')": "Utilizo_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "('P7_a_5 ', 'Realizo experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.')": "Realizo_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "('P7_a_6 ', 'Desenvolvo/cuido da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.')": "Desenvolvo_cuido_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "('P7_a_7 ', 'Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados, Data Warehouses, Data Marts etc.')": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "('P7_a_8 ', 'Desenvolvo/cuido da manutenção de planilhas para atender as áreas de negócio.')": "Desenvolvo_cuido_da_manutencao_de_planilhas_para_atender_as_areas_de_negocio",
    "('P7_a_9 ', 'Utilizo ferramentas avançadas de estatística como SASS, PSS, Stata etc')": "Utilizo_ferramentas_avancadas_de_estatistica_como_SASS_PSS_Stata_etc",
    "('P7_b ', 'Quais as ferramentas/tecnologias de ETL que você utiliza no trabalho como Data Analyst?')": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Analyst_ETL",
    "('P7_c ', 'Sua empresa utiliza alguma das ferramentas listadas para dar mais autonomia em análise de dados para as áreas de negócio?')": "Sua_empresa_utiliza_alguma_das_ferramentas_listadas_para_dar_mais_autonomia_em_analise_de_dados_para_as_areas_de_negocio",
    "('P7_c_1 ', 'Ferramentas de AutoML como H2O.ai, Data Robot, BigML etc.')": "Ferramentas_de_AutoML_como_H2O_ai_Data_Robot_BigML_etc",
    "('P7_c_2 ', 'Point and Click Analytics como Alteryx, Knime, Rapidminer etc.')": "Point_and_Click_Analytics_como_Alteryx_Knime_Rapidminer_etc",
    "('P7_c_3 ', 'Product metricts & Insights como Mixpanel, Amplitude, Adobe Analytics.')": "Product_metricts_Insights_como_Mixpanel_Amplitude_Adobe_Analytics",
    "('P7_c_4 ', 'Ferramentas de análise dentro de ferramentas de CRM como Salesforce Einstein Anaytics ou Zendesk dashboards.')": "Ferramentas_de_analise_dentro_de_ferramentas_de_CRM_como_Salesforce_Einstein_Anaytics_ou_Zendesk_dashboards",
    "('P7_c_5 ', 'Minha empresa não utiliza essas ferramentas.')": "Minha_empresa_nao_utiliza_essas_ferramentas",
    "('P7_c_6 ', 'Não sei informar.')": "Nao_sei_informar",
    "('P7_d ', 'Em qual das opções abaixo você gasta a maior parte do seu tempo de trabalho?')": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_de_trabalho",
    "('P7_d_1 ', 'Processando e analisando dados utilizando linguagens de programação como Python, R etc.')": "Processando_e_analisando_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "('P7_d_2 ', 'Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.')": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_P7d",
    "('P8_d_7 ', 'Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc.')": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_P8d",
    "('P7_d_4 ', 'Utilizando API's para extrair dados e complementar minhas análises.')": "Utilizando_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "('P7_d_5 ', 'Realizando experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.')": "Realizando_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "('P7_d_6 ', 'Desenvolvendo/cuidando da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.')": "Desenvolvendo_cuidando_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "('P7_d_7 ', 'Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados, Data Warehouses, Data Marts etc.')": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "('P7_d_8 ', 'Desenvolvendo/cuidando da manutenção de planilhas do Excel ou Google Sheets para atender as áreas de negócio.')": "Desenvolvendo_cuidando_da_manutencao_de_planilhas_do_Excel_ou_Google_Sheets_para_atender_as_areas_de_negocio",
    "('P7_d_9 ', 'Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.')": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_P7d",
    "('P8_d_8 ', 'Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.')": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_P8d",
    "('P8_a ', 'Quais das opções abaixo fazem parte da sua rotina no trabalho atual com ciência de dados?')": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_ciencia_de_dados",
    "('P8_a_1 ', 'Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.')": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_P8a",
    "('P8_d_1 ', 'Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.')": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_P8d",
    "('P8_a_2 ', 'Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.')": "Sou_responsavel_pela_coleta_e_limpeza_dos_dados_que_uso_para_analise_e_modelagem",
    "('P8_a_3 ', 'Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.')": "Sou_responsavel_por_entrar_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "('P8_a_4 ', 'Desenvolvo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).')": "Desenvolvo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "('P8_a_5 ', 'Sou responsável por colocar modelos em produção, criar os pipelines de dados, APIs de consumo e monitoramento.')": "Sou_responsavel_por_colocar_modelos_em_producao_criar_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "('P8_a_6 ', 'Cuido da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.')": "Cuido_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "('P8_a_8 ', 'Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises estatísticas e ajustar modelos.')": "Utilizo_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_estatisticas_e_ajustar_modelos",
    "('P8_a_9 ', 'Crio e dou manutenção em ETLs, DAGs e automações de pipelines de dados.')": "Crio_e_dou_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "('P8_a_10 ', 'Crio e gerencio soluções de Feature Store e cultura de MLOps.')": "Crio_e_gerencio_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "('P8_a_11 ', 'Sou responsável por criar e manter a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)')": "Sou_responsavel_por_criar_e_manter_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "('P8_a_12 ', 'Treino e aplico LLM's para solucionar problemas de negócio.')": "Treino_e_aplico_LLM_s_para_solucionar_problemas_de_negocio",
    "('P8_b ', 'Quais as técnicas e métodos listados abaixo você costuma utilizar no trabalho?')": "Quais_as_tecnicas_e_metodos_listados_abaixo_voce_costuma_utilizar_no_trabalho",
    "('P8_b_1 ', 'Utilizo modelos de regressão (linear, logística, GLM)')": "Utilizo_modelos_de_regressao_linear_logistica_GLM",
    "('P8_b_2 ', 'Utilizo redes neurais ou modelos baseados em árvore para criar modelos de classificação')": "Utilizo_redes_neurais_ou_modelos_baseados_em_arvore_para_criar_modelos_de_classificacao",
    "('P8_b_3 ', 'Desenvolvo sistemas de recomendação (RecSys)')": "Desenvolvo_sistemas_de_recomendacao_RecSys",
    "('P8_b_4 ', 'Utilizo métodos estatísticos Bayesianos para analisar dados')": "Utilizo_metodos_estatisticos_Bayesianos_para_analisar_dados",
    "('P8_b_5 ', 'Utilizo técnicas de NLP (Natural Language Processing) para análisar dados não-estruturados')": "Utilizo_tecnicas_de_NLP_Natural_Language_Processing_para_analisar_dados_nao_estruturados",
    "('P8_b_6 ', 'Utilizo métodos estatísticos clássicos (Testes de hipótese, análise multivariada, sobrevivência, dados longitudinais, inferência estatistica) para analisar dados')": "Utilizo_metodos_estatisticos_classicos_Testes_de_hipotese_analise_multivariada_sobrevivencia_dados_longitudinais_inferencia_estatistica_para_analisar_dados",
    "('P8_b_7 ', 'Utilizo cadeias de Markov ou HMM's para realizar análises de dados')": "Utilizo_cadeias_de_Markov_ou_HMM_s_para_realizar_analises_de_dados",
    "('P8_b_8 ', 'Desenvolvo técnicas de Clusterização (K-means, Spectral, DBScan etc)')": "Desenvolvo_tecnicas_de_Clusterizacao_K_means_Spectral_DBScan_etc",
    "('P8_b_9 ', 'Realizo previsões através de modelos de Séries Temporais (Time Series)')": "Realizo_previsoes_atraves_de_modelos_de_Series_Temporais_Time_Series",
    "('P8_b_10 ', 'Utilizo modelos de Reinforcement Learning (aprendizado por reforço)')": "Utilizo_modelos_de_Reinforcement_Learning_aprendizado_por_reforco",
    "('P8_b_11 ', 'Utilizo modelos de Machine Learning para detecção de fraude')": "Utilizo_modelos_de_Machine_Learning_para_deteccao_de_fraude",
    "('P8_b_12 ', 'Utilizo métodos de Visão Computacional')": "Utilizo_metodos_de_Visao_Computacional",
    "('P8_b_13 ', 'Utilizo modelos de Detecção de Churn')": "Utilizo_modelos_de_Deteccao_de_Churn",
    "('P8_b_14 ', 'Utilizo LLM's para solucionar problemas de negócio')": "Utilizo_LLM_s_para_solucionar_problemas_de_negocio",
    "('P8_3 ', 'Quais dessas tecnologias fazem parte do seu dia a dia como cientista de dados?')": "Quais_dessas_tecnologias_fazem_parte_do_seu_dia_a_dia_como_cientista_de_dados",
    "('P8_c_1 ', 'Ferramentas de BI (PowerBI, Looker, Tableau, Qlik etc)')": "Ferramentas_de_BI_PowerBI_Looker_Tableau_Qlik_etc",
    "('P8_c_2 ', 'Planilhas (Excel, Google Sheets etc)')": "Planilhas_Excel_Google_Sheets_etc",
    "('P8_c_3 ', 'Ambientes de desenvolvimento local (R-studio, JupyterLab, Anaconda)')": "Ambientes_de_desenvolvimento_local_R_studio_JupyterLab_Anaconda",
    "('P8_c_4 ', 'Ambientes de desenvolvimento na nuvem (Google Colab, AWS Sagemaker, Kaggle Notebooks etc)')": "Ambientes_de_desenvolvimento_na_nuvem_Google_Colab_AWS_Sagemaker_Kaggle_Notebooks_etc",
    "('P8_c_5 ', 'Ferramentas de AutoML (Datarobot, H2O, Auto-Keras etc)')": "Ferramentas_de_AutoML_Datarobot_H2O_Auto_Keras_etc",
    "('P8_c_6 ', 'Ferramentas de ETL (Apache Airflow, NiFi, Stitch, Fivetran, Pentaho etc)')": "Ferramentas_de_ETL_Apache_Airflow_NiFi_Stitch_Fivetran_Pentaho_etc",
    "('P8_c_7 ', 'Plataformas de Machine Learning (TensorFlow, Azure Machine Learning, Kubeflow etc)')": "Plataformas_de_Machine_Learning_TensorFlow_Azure_Machine_Learning_Kubeflow_etc",
    "('P8_c_8 ', 'Feature Store (Feast, Hopsworks, AWS Feature Store, Databricks Feature Store etc)')": "Feature_Store_Feast_Hopsworks_AWS_Feature_Store_Databricks_Feature_Store_etc",
    "('P8_c_9 ', 'Sistemas de controle de versão (Github, DVC, Neptune, Gitlab etc)')": "Sistemas_de_controle_de_versao_Github_DVC_Neptune_Gitlab_etc",
    "('P8_c_10 ', 'Plataformas de Data Apps (Streamlit, Shiny, Plotly Dash etc)')": "Plataformas_de_Data_Apps_Streamlit_Shiny_Plotly_Dash_etc",
    "('P8_c_11 ', 'Ferramentas de estatística avançada como SPSS, SAS, etc.')": "Ferramentas_de_estatistica_avancada_como_SPSS_SAS_etc",
    "('P8_d ', 'Em qual das opções abaixo você gasta a maior parte do seu tempo no trabalho?')": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_no_trabalho",
    "('P8_d_2 ', 'Coletando e limpando os dados que uso para análise e modelagem.')": "Coletando_e_limpando_os_dados_que_uso_para_analise_e_modelagem",
    "('P8_d_3 ', 'Entrando em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.')": "Entrando_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "('P8_d_4 ', 'Desenvolvendo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).')": "Desenvolvendo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "('P8_d_5 ', 'Colocando modelos em produção, criando os pipelines de dados, APIs de consumo e monitoramento.')": "Colocando_modelos_em_producao_criando_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "('P8_d_6 ', 'Cuidando da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.')": "Cuidando_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "('P8_d_9 ', 'Criando e dando manutenção em ETLs, DAGs e automações de pipelines de dados.')": "Criando_e_dando_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "('P8_d_10 ', 'Criando e gerenciando soluções de Feature Store e cultura de MLOps.')": "Criando_e_gerenciando_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "('P8_d_11 ', 'Criando e mantendo a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)')": "Criando_e_mantendo_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "('P8_d_12 ', 'Treinando e aplicando LLM's para solucionar problemas de negócio.')": "Treinando_e_aplicando_LLM_s_para_solucionar_problemas_de_negocio",
}

In [ ]:
mapa_2024 = {
    "0.a_token": "id",
    "0.d_data/hora_envio": "data_hora_envio",
    "1.a_idade": "Idade",
    "1.a.1_faixa_idade": "Faixa_idade",
    "1.b_genero": "Genero",
    "1.c_cor/raca/etnia": "Cor_raca_etnia",
    "1.d_pcd": "PCD",
    "1.e_experiencia_profissional_prejudicada": "experiencia_profissional_prejudicada",
    "1.e.1_Não acredito que minha experiência profissional seja afetada": "Nao_acredito_que_minha_experiencia_profissional_seja_afetada",
    "1.e.2_Sim, devido a minha Cor/Raça/Etnia": "Experiencia_prejudicada_devido_a_minha_Cor_Raca_Etnia",
    "1.e.3_Sim, devido a minha identidade de gênero": "Experiencia_prejudicada_devido_a_minha_identidade_de_genero",
    "1.e.4_Sim, devido ao fato de ser PCD": "Experiencia_prejudicada_devido_ao_fato_de_ser_PCD",
    "1.f_aspectos_prejudicados": "aspectos_prejudicados",
    "1.f.1_Quantidade de oportunidades de emprego/vagas recebidas": "Quantidade_de_oportunidades_de_emprego_vagas_recebidas",
    "1.f.2_Senioridade das vagas recebidas em relação à sua experiência": "Senioridade_das_vagas_recebidas_em_relacao_a_sua_experiencia",
    "1.f.3_Aprovação em processos seletivos/entrevistas": "Aprovacao_em_processos_seletivos_entrevistas",
    "1.f.4_Oportunidades de progressão de carreira": "Oportunidades_de_progressao_de_carreira",
    "1.f.5_Velocidade de progressão de carreira": "Velocidade_de_progressao_de_carreira",
    "1.f.6_Nível de cobrança no trabalho/Stress no trabalho": "Nivel_de_cobranca_no_trabalho_Stress_no_trabalho",
    "1.f.7_Atenção dada pelas pessoas diante das minhas opiniões e ideias": "Atencao_dada_diante_das_minhas_opinioes_e_ideias",
    "1.f.8_Relação com outras pessoas da empresa, em momentos de trabalho": "Relacao_com_outros_membros_da_empresa_em_momentos_de_trabalho",
    "1.f.9_Relação com outras pessoas da empresa, em momentos de integração e outros momentos fora do trabalho": "Relacao_com_outros_membros_da_empresa_em_momentos_de_integracao_e_outros_momentos_fora_do_trabalho",
    "1.g_vive_no_brasil": "vive_no_brasil",
    "1.h_pais_onde_mora": "pais_onde_mora",
    "1.i_estado_onde_mora": "Estado_onde_mora",
    "1.i.2_regiao_onde_mora": "Regiao_onde_mora",
    "1.k_estado_de_origem": "Estado_de_origem",
    "1.k.1_uf_de_origem": "uf_de_origem",
    "1.k.2_regiao_de_origem": "Regiao_de_origem",
    "1.j_vive_no_estado_de_formacao": "vive_no_estado_de_formacao",
    "1.l_nivel_de_ensino": "Nivel_de_Ensino",
    "1.m_área_de_formação": "Area_de_Formacao",
    "2.a_situação_de_trabalho": "Qual_sua_situacao_atual_de_trabalho",
    "2.b_setor": "Setor",
    "2.c_numero_de_funcionarios": "Numero_de_Funcionarios",
    "2.d_atua_como_gestor": "Gestor",
    "2.e_cargo_como_gestor": "Cargo_como_Gestor",
    "2.f_cargo_atual": "Cargo_Atual",
    "2.g_nivel": "Nivel",
    "2.h_faixa_salarial": "Faixa_salarial",
    "2.i_tempo_de_experiencia_em_dados": "Quanto_tempo_de_experiencia_na_area_de_dados_voce_tem",
    "2.j_tempo_de_experiencia_em_ti": "Quanto_tempo_de_experiencia_na_area_de_TI_Engenharia_de_Software_voce_teve_antes_de_comecar_a_trabalhar_na_area_de_dados",
    "2.k_satisfeito_atualmente": "Voce_esta_satisfeito_na_sua_empresa_atual",
    "2.l_motivo_insatisfacao": "Qual_o_principal_motivo_da_sua_insatisfacao_com_a_empresa_atual",
    "2.l.1_Remuneração/Salário": "Salario_atual_nao_corresponde_ao_mercado",
    "2.l.2_Benefícios": "Gostaria_de_receber_mais_beneficios",
    "2.l.3_Propósito do trabalho e da empresa": "Proposito_do_trabalho_e_da_empresa_2l",
    "2.o.3_Propósito do trabalho e da empresa": "Proposito_do_trabalho_e_da_empresa_2o",
    "2.l.4_Flexibilidade de trabalho remoto": "Flexibilidade_de_trabalho_remoto_2l",
    "2.o.4_Flexibilidade de trabalho remoto": "Flexibilidade_de_trabalho_remoto_2o",
    "2.l.5_Ambiente e clima de trabalho": "O_clima_de_trabalho_ambiente_nao_e_bom",
    "2.l.6_Oportunidade de aprendizado e trabalhar com referências": "Oportunidade_de_aprendizado_e_trabalhar_com_referencias",
    "2.l.7_Oportunidades de crescimento": "Falta_de_oportunidade_de_crescimento_no_emprego_atual",
    "2.l.8_Maturidade da empresa em termos de tecnologia e dados": "Falta_de_maturidade_analitica_na_empresa",
    "2.l.9_Relação com os gestores e líderes": "Nao_tenho_uma_boa_relacao_com_meu_lider_gestor",
    "2.l.10_Reputação que a empresa tem no mercado": "Reputacao_que_a_empresa_tem_no_mercado_2l",
    "2.o.10_Reputação que a empresa tem no mercado": "Reputacao_que_a_empresa_tem_no_mercado_2o",
    "2.l.11_Gostaria de trabalhar em outra área": "Gostaria_de_trabalhar_em_em_outra_area_de_atuacao",
    "2.m_participou_de_entrevistas_ultimos_6m": "Voce_participou_de_entrevistas_de_emprego_nos_ultimos_6_meses",
    "2.n_planos_de_mudar_de_emprego_6m": "Voce_pretende_mudar_de_emprego_nos_proximos_6_meses",
    "2.o_criterios_para_escolha_de_emprego": "Quais_os_principais_criterios_que_voce_leva_em_consideracao_no_momento_de_decidir_onde_trabalhar",
    "2.o.1_Remuneração/Salário": "Remuneracao_Salario",
    "2.o.2_Benefícios": "Beneficios",
    "2.o.5_Ambiente e clima de trabalho": "Ambiente_e_clima_de_trabalho",
    "2.o.6_Oportunidade de aprendizado e trabalhar com referências": "Oportunidade_de_aprendizado_e_trabalhar_com_referencias_na_area",
    "2.o.7_Plano de carreira e oportunidades de crescimento": "Plano_de_carreira_e_oportunidades_de_crescimento_profissional",
    "2.o.8_Maturidade da empresa em termos de tecnologia e dados": "Maturidade_da_empresa_em_termos_de_tecnologia_e_dados",
    "2.o.9_Qualidade dos gestores e líderes": "Qualidade_dos_gestores_e_lideres",
    "2.q_empresa_passou_por_layoff_em_2024": "Empresa_que_trabaha_passou_por_layoff_em_2023",
    "2.r_modelo_de_trabalho_atual": "Atualmente_qual_a_sua_forma_de_trabalho",
    "2.s_modelo_de_trabalho_ideal": "Qual_a_forma_de_trabalho_ideal_para_voce",
    "2.t_atitude_em_caso_de_retorno_presencial": "Caso_sua_empresa_decida_pelo_modelo_100_presencial_qual_sera_sua_atitude",
    "3.a_numero_de_pessoas_em_dados": "Qual_o_numero_aproximado_de_pessoas_que_atuam_com_dados_na_sua_empresa_hoje",
    "3.b_cargos_no_time_de_dados_da_empresa": "Quais_desses_papeis_cargos_fazem_parte_do_time_ou_chapter_de_dados_da_sua_empresa",
    "3.b.1_Analytics Engineer": "Analytics_Engineer",
    "3.b.2_Engenharia de Dados/Data Engineer": "Engenharia_de_Dados_Data_Engineer",
    "3.b.3_Analista de Dados/Data Analyst": "Analista_de_Dados_Data_Analyst",
    "3.b.4_Cientista de Dados/Data Scientist": "Cientista_de_Dados_Data_Scientist",
    "3.b.5_Database Administrator/DBA": "Database_Administrator_DBA",
    "3.b.6_Analista de Business Intelligence/BI": "Analista_de_Business_Intelligence_BI",
    "3.b.7_Arquiteto de Dados/Data Architect": "Arquiteto_de_Dados_Data_Architect",
    "3.b.8_Data Product Manager/DPM": "Data_Product_Manager_DPM",
    "3.b.9_Business Analyst": "Business_Analyst",
    "3.b.10_ML Engineer/AI Engineer": "ML_Engineer_AI_Engineer",
    "3.c_responsabilidades_como_gestor": "Quais_dessas_responsabilidades_fazem_parte_da_sua_rotina_atual_de_trabalho_como_gestor",
    "3.c.1_Pensar na visão de longo prazo de dados": "Pensar_na_visao_de_longo_prazo_de_dados_da_empresa_e_fortalecimento_da_cultura_analitica_da_companhia",
    "3.c.2_Organização de treinamentos e iniciativas": "Organizacao_de_treinamentos_e_iniciativas_com_o_objetivo_de_aumentar_a_maturidade_analitica_das_areas_de_negocios",
    "3.c.3_Atração, seleção e contratação": "Atracao_selecao_e_contratacao_de_talentos_para_o_time_de_dados",
    "3.c.4_Decisão sobre contratação de ferramentas": "Decisao_sobre_contratacao_de_ferramentas_e_tecnologias_relacionadas_a_dados",
    "3.c.5_gestor da equipe de engenharia de dados": "Sou_gestor_da_equipe_responsavel_pela_engenharia_de_dados_e_por_manter_o_Data_Lake_da_empresa_como_fonte_unica_dos_dados_garantindo_a_qualidade_e_confiabilidade_da_informacao",
    "3.c.6_gestor da equipe de estudos, relatórios": "Sou_gestor_da_equipe_responsavel_pela_entrega_de_dados_estudos_relatorios_e_dashboards_para_as_areas_de_negocio_da_empresa",
    "3.c.7_gestor da equipe de Inteligência Artificial e Machine Learning": "Sou_gestor_da_equipe_responsavel_por_iniciativas_e_projetos_envolvendo_Inteligencia_Artificial_e_Machine_Learning",
    "3.c.8_Apesar de ser gestor ainda atuo na parte técnica": "Apesar_de_ser_gestor_ainda_atuo_na_parte_tecnica_construindo_solucoes_analises_modelos_etc",
    "3.c.9_Gestão de projetos de dados": "Gestao_de_projetos_de_dados_cuidando_das_etapas_equipes_envolvidas_atingimento_dos_objetivos_etc",
    "3.c.10_Gestão de produtos de dados": "Gestao_de_produtos_de_dados_cuidando_da_visao_dos_produtos_backlog_feedback_de_usuarios_etc",
    "3.c.11_Gestão de pessoas": "Gestao_de_pessoas_apoio_no_desenvolvimento_das_pessoas_evolucao_de_carreira",
    "3.d_desafios_como_gestor": "Quais_sao_os_3_maiores_desafios_que_voce_tem_como_gestor_no_atual_momento",
    "3.d.1_Contratar talentos": "a_Contratar_novos_talentos",
    "3.d.2_Reter talentos": "b_Reter_talentos",
    "3.d.3_Convencer a empresa a aumentar investimentos": "c_Convencer_a_empresa_a_aumentar_os_investimentos_na_area_de_dados",
    "3.d.4_Gestão de equipes no ambiente remoto": "d_Gestao_de_equipes_no_ambiente_remoto",
    "3.d.5_Gestão de projetos envolvendo áreas multidisciplinares": "e_Gestao_de_projetos_envolvendo_areas_multidisciplinares_da_empresa",
    "3.d.6_Organizar as informações com qualidade e confiabilidade": "f_Organizar_as_informacoes_e_garantir_a_qualidade_e_confiabilidade",
    "3.d.7_Processar e armazenar um alto volume de dados": "g_Conseguir_processar_e_armazenar_um_alto_volume_de_dados",
    "3.d.8_Gerar valor para as áreas de negócios": "h_Conseguir_gerar_valor_para_as_areas_de_negocios_atraves_de_estudos_e_experimentos",
    "3.d.9_Desenvolver e manter modelos Machine Learning em produção": "i_Desenvolver_e_manter_modelos_Machine_Learning_em_producao",
    "3.d.10_Gerenciar a expectativa das áreas": "j_Gerenciar_a_expectativa_das_areas_de_negocio_em_relacao_as_entregas_das_equipes_de_dados",
    "3.d.11_Garantir a manutenção dos projetos e modelos em produção": "k_Garantir_a_manutencao_dos_projetos_e_modelos_em_producao_em_meio_ao_crescimento_da_empresa",
    "3.d.12_Conseguir levar inovação para a empresa": "Conseguir_levar_inovacao_para_a_empresa_atraves_dos_dados",
    "3.d.13_Garantir (ROI) em projetos de dados": "Garantir_retorno_do_investimento_ROI_em_projetos_de_dados",
    "3.d.14_Dividir o tempo entre entregas técnicas e gestão": "Dividir_o_tempo_entre_entregas_tecnicas_e_gestao",
    "3.e_ai_generativa_e_llm_é_uma_prioridade?": "AI_Generativa_e_uma_prioridade_em_sua_empresa",
    "3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "Tipos_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "3.f.1 Colaboradores usando AI generativa de forma independente e descentralizada": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_3f",
    "4.l.1 Colaboradores usando AI generativa de forma independente e descentralizada": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_4l",
    "3.f.2 Direcionamento centralizado do uso de AI generativa": "Direcionamento_centralizado_do_uso_de_AI_generativa_3f",
    "4.l.2 Direcionamento centralizado do uso de AI generativa": "Direcionamento_centralizado_do_uso_de_AI_generativa_4l",
    "3.f.3 Desenvolvedores utilizando Copilots": "Desenvolvedores_utilizando_Copilots_3f",
    "4.l.3 Desenvolvedores utilizando Copilots": "Desenvolvedores_utilizando_Copilots_4l",
    "3.f.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos",
    "3.f.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_3f",
    "4.l.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_4l",
    "3.f.6 IA Generativa e LLMs como principal frente do negócio": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_3f",
    "4.l.6 IA Generativa e LLMs como principal frente do negócio": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_4l",
    "3.f.7 IA Generativa e LLMs não é prioridade": "IA_Generativa_e_LLMs_nao_e_prioridade_3f",
    "4.l.7 IA Generativa e LLMs não é prioridade": "IA_Generativa_e_LLMs_nao_e_prioridade_4l",
    "3.f.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_3f",
    "4.l.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_4l",
    "3.g_motivos_para_não_usar_ai_generativa_e_llm": "Motivos_que_levam_a_empresa_a_nao_usar_AI_Genrativa_e_LLMs",
    "3.g.1 Falta de compreensão dos casos de uso": "Falta_de_compreensao_dos_casos_de_uso",
    "3.g.2 Falta de confiabilidade das saídas (alucinação dos modelos)": "Falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos",
    "3.g.3 Incerteza em relação a regulamentação": "Incerteza_em_relacao_a_regulamentacao",
    "3.g.4 Preocupações com segurança e privacidade de dados": "Preocupacoes_com_seguranca_e_privacidade_de_dados",
    "3.g.5 Retorno sobre investimento (ROI) não comprovado de IA Generativa": "Retorno_sobre_investimento_ROI_nao_comprovado_de_IA_Generativa",
    "3.g.6 Dados da empresa não estão prontos para uso de IA Generativa": "Dados_da_empresa_nao_estao_prontos_para_uso_de_IA_Generativa",
    "3.g.7 Falta de expertise ou falta de recursos": "Falta_de_expertise_ou_falta_de_recursos",
    "3.g.8 Alta direção da empresa não vê valor ou não vê como prioridade": "Alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade",
    "3.g.9 Preocupações com propriedade intelectual": "Preocupacoes_com_propriedade_intelectual",
    "4.a_funcao_de_atuacao": "Mesmo_que_esse_nao_seja_seu_cargo_formal_voce_considera_que_sua_atuacao_no_dia_a_dia_reflete_alguma_das_opcoes_listadas_abaixo",
    "4.a.1_atuacao_em_dados": "Atuacao",
    "4.b_fontes_de_dados_(dia_a_dia)": "Quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho",
    "4.b.1_Dados relacionais (estruturados em bancos SQL)": "Dados_relacionais_estruturados_em_bancos_SQL_4b",
    "4.c.1_Dados relacionais (estruturados em bancos SQL)": "Dados_relacionais_estruturados_em_bancos_SQL_4c",
    "4.b.2_Dados armazenados em bancos NoSQL": "Dados_armazenados_em_bancos_NoSQL_4b",
    "4.c.2_Dados armazenados em bancos NoSQL": "Dados_armazenados_em_bancos_NoSQL_4c",
    "4.b.3_Imagens": "Imagens_4b",
    "4.c.3_Imagens": "Imagens_4c",
    "4.b.4_Textos/Documentos": "Textos_Documentos_4b",
    "4.c.4_Textos/Documentos": "Textos_Documentos_4c",
    "4.b.5_Vídeos": "Videos_4b",
    "4.c.5_Vídeos": "Videos_4c",
    "4.b.6_Áudios": "Audios_4b",
    "4.c.6_Áudios": "Audios_4c",
    "4.b.7_Planilhas": "Planilhas_4b",
    "4.c.7_Planilhas": "Planilhas_4c",
    "4.b.8_Dados georeferenciados": "Dados_georeferenciados_4b",
    "4.c.8_Dados georeferenciados": "Dados_georeferenciados_4c",
    "4.c_fonte_de_dado_mais_usada": "Entre_as_fontes_de_dados_listadas_quais_voce_utiliza_na_maior_parte_do_tempo",
    "4.d_linguagem_de_programacao_(dia_a_dia)": "Quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho_Linguagem",
    "4.d.1_SQL": "SQL_Linguagem",
    "4.d.2_R": "R_Linguagem",
    "4.d.3_Python": "Python_Linguagem",
    "4.d.4_C/C++/C#": "C_C_C_Linguagem",
    "4.d.5_.NET": "NET_Linguagem",
    "4.d.6_Java": "Java_Linguagem",
    "4.d.7_Julia": "Julia_Linguagem",
    "4.d.8_SAS/Stata": "SAS_Stata_Linguagem",
    "4.d.9_Visual Basic/VBA": "Visual_Basic_VBA_Linguagem",
    "4.d.10_Scala": "Scala_Linguagem",
    "4.d.11_Matlab": "Matlab_Linguagem",
    "4.d.12_Rust": "Rust_Linguagem",
    "4.d.13_PHP": "PHP_Linguagem",
    "4.d.14_JavaScript": "JavaScript_Linguagem",
    "4.d.15_Não utilizo nenhuma das linguagens listadas": "Nao_utilizo_nenhuma_linguagem_Linguagem",
    "4.e_linguagem_mais_usada": "Entre_as_linguagens_listadas_abaixo_qual_e_a_que_voce_mais_utiliza_no_trabalho",
    "4.f_linguagem_preferida": "Entre_as_linguagens_listadas_abaixo_qual_e_a_sua_preferida",
    "4.g_banco_de_dados_(dia_a_dia)": "Quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho_BD",
    "4.g.1_MySQL": "MySQL_BD",
    "4.g.2_Oracle": "Oracle_BD",
    "4.g.3_SQL SERVER": "SQL_SERVER_BD",
    "4.g.4_Amazon Aurora ou RDS": "Amazon_Aurora_ou_RDS_BD",
    "4.g.5_DynamoDB": "DynamoDB_BD",
    "4.g.6_CoachDB": "CoachDB_BD",
    "4.g.7_Cassandra": "Cassandra_BD",
    "4.g.8_MongoDB": "MongoDB_BD",
    "4.g.9_MariaDB": "MariaDB_BD",
    "4.g.10_Datomic": "Datomic_BD",
    "4.g.11_S3": "S3_BD",
    "4.g.12_PostgreSQL": "PostgreSQL_BD",
    "4.g.13_ElasticSearch": "ElasticSearch_BD",
    "4.g.14_DB2": "DB2_BD",
    "4.g.15_Microsoft Access": "Microsoft_Access_BD",
    "4.g.16_SQLite": "SQLite_BD",
    "4.g.17_Sybase": "Sybase_BD",
    "4.g.18_Firebase": "Firebase_BD",
    "4.g.19_Vertica": "Vertica_BD",
    "4.g.20_Redis": "Redis_BD",
    "4.g.21_Neo4J": "Neo4J_BD",
    "4.g.22_Google BigQuery": "Google_BigQuery_BD",
    "4.g.23_Google Firestore": "Google_Firestore_BD",
    "4.g.24_Amazon Redshift": "Amazon_Redshift_BD",
    "4.g.25_Amazon Athena": "Amazon_Athena_BD",
    "4.g.26_Snowflake": "Snowflake_BD",
    "4.g.27_Databricks": "Databricks_BD",
    "6.b.20_Databricks": "Databricks_6b",
    "7.b.20_Databricks": "Databricks_ETL",
    "4.g.28_HBase": "HBase_BD",
    "4.g.29_Presto": "Presto_BD",
    "4.g.30_Splunk": "Splunk_BD",
    "4.g.31_SAP HANA": "SAP_HANA_BD",
    "4.g.32_Hive": "Hive_BD",
    "4.g.33_Firebird": "Firebird_BD",
    "4.h_cloud_(dia_a_dia)": "Dentre_as_opcoes_listadas_qual_sua_Cloud_preferida_cloud",
    "4.h.1_Amazon Web Services (AWS)": "Amazon_Web_Services_AWS_cloud",
    "4.h.2_Google Cloud (GCP)": "Google_Cloud_GCP_cloud",
    "4.h.3_Azure (Microsoft)": "Azure_Microsoft_cloud",
    "4.h.4_Oracle Cloud": "Oracle_Cloud_cloud",
    "4.h.5_IBM": "IBM_cloud",
    "4.h.6_Servidores On Premise/Não utilizamos Cloud": "Servidores_On_Premise_Nao_utilizamos_Cloud_cloud",
    "4.h.7_Cloud Própria": "Cloud_Propria_cloud",
    "4.i_cloud_preferida": "Cloud_preferida",
    "4.j_ferramenta_de_bi_(dia_a_dia)": "Ferramenta_de_BI_utilizada_no_dia_a_dia_BI",
    "4.j.1_Microsoft PowerBI": "Microsoft_PowerBI_BI",
    "4.j.2_Qlik View/Qlik Sense": "Qlik_View_Qlik_Sense_BI",
    "4.j.3_Tableau": "Tableau_BI",
    "4.j.4_Metabase": "Metabase_BI",
    "4.j.5_Superset": "Superset_BI",
    "4.j.6_Redash": "Redash_BI",
    "4.j.7_Looker": "Looker_BI",
    "4.j.8_Looker Studio(Google Data Studio)": "Looker_Studio_Google_Data_Studio_BI",
    "4.j.9_Amazon Quicksight": "Amazon_Quicksight_BI",
    "4.j.10_Alteryx": "Alteryx_BI",
    "6.b.9_Alteryx": "Alteryx_6b",
    "7.b.9_Alteryx": "Alteryx_ETL",
    "4.j.11_SAP Business Objects/SAP Analytics": "SAP_Business_Objects_SAP_Analytics_BI",
    "4.j.12_Oracle Business Intelligence": "Oracle_Business_Intelligence_BI",
    "4.j.13_Salesforce/Einstein Analytics": "Salesforce_Einstein_Analytics_BI",
    "4.j.14_SAS Visual Analytics": "SAS_Visual_Analytics_BI",
    "4.j.15_Grafana": "Grafana_BI",
    "4.j.16_Pentaho": "Pentaho_BI",
    "6.b.8_Pentaho": "Pentaho_6b",
    "7.b.8_Pentaho": "Pentaho_ETL",
    "4.j.17_Fazemos todas as análises utilizando apenas Excel ou planilhas do google": "Fazemos_todas_as_analises_utilizando_apenas_Excel_ou_planilhas_do_google_BI",
    "4.j.18_Não utilizo nenhuma ferramenta de BI no trabalho": "Nao_utilizo_nenhuma_ferramenta_de_BI_no_trabalho_BI",
    "4.k_ferramenta_de_bi_preferida": "Qual_sua_ferramenta_de_BI_preferida",
    "4.l_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "Qual_o_tipo_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "4.l.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos_para_os_clientes_finais",
    "4.m_usa_chatgpt_ou_copilot_no_trabalho?": "Utiliza_ChatGPT_ou_LLMs_no_trabalho",
    "4.m.1 Não uso soluções de AI Generativa com foco em produtividade": "Nao_uso_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.m.2 Uso soluções gratuitas de AI Generativa com foco em produtividade": "Uso_solucoes_gratuitas_de_AI_Generativa_com_foco_em_produtividade",
    "4.m.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade": "Uso_e_pago_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.m.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade": "A_empresa_que_trabalho_paga_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.m.5 Uso soluções do tipo Copilot": "Uso_solucoes_do_tipo_Copilot",
    "5.a_objetivo_na_area_de_dados": "Qual_seu_objetivo_na_area_de_dados",
    "5.b_oportunidade_buscada": "Qual_oportunidade_voce_esta_buscando",
    "5.c_tempo_em_busca_de_oportunidade": "Ha_quanto_tempo_voce_busca_uma_oportunidade_na_area_de_dados",
    "5.d_experiencia_em_processos_seletivos": "Como_tem_sido_a_busca_por_um_emprego_na_area_de_dados",
    "6.a_rotina_como_de": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_como_engenheiro_de_dados",
    "6.a.1_Desenvolvo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "Desenvolvo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "6.a.2_Realizo construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.": "Realizo_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "6.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_6a",
    "7.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_7a",
    "6.a.4_Atuo na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "Atuo_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "6.a.5_Modelo soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "Modelo_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "6.a.6_Desenvolvo/cuido da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "Desenvolvo_cuido_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "6.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "6.a.8_Cuido da qualidade dos dados, metadados e dicionário de dados.": "Cuido_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "6.a.9_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_6a",
    "6.h.9_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_6h",
    "7.a.10_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_7a",
    "7.d.10_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_7d",
    "6.b_ferramentas_etl_de": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Engineer",
    "6.b.1_Scripts Python": "Scripts_Python_6b",
    "7.b.1_Scripts Python": "Scripts_Python_ETL",
    "6.b.2_SQL & Stored Procedures": "SQL_Stored_Procedures_6b",
    "7.b.2_SQL & Stored Procedures": "SQL_Stored_Procedures_ETL",
    "6.b.3_Apache Airflow": "Apache_Airflow_6b",
    "7.b.3_Apache Airflow": "Apache_Airflow_ETL",
    "6.b.4_Apache NiFi": "Apache_NiFi_6b",
    "7.b.4_Apache NiFi": "Apache_NiFi_ETL",
    "6.b.5_Luigi": "Luigi_6b",
    "7.b.5_Luigi": "Luigi_ETL",
    "6.b.6_AWS Glue": "AWS_Glue_6b",
    "7.b.6_AWS Glue": "AWS_Glue_ETL",
    "6.b.7_Talend": "Talend_6b",
    "7.b.7_Talend": "Talend_ETL",
    "6.b.10_Stitch": "Stitch_6b",
    "7.b.10_Stitch": "Stitch_ETL",
    "6.b.11_Fivetran": "Fivetran_6b",
    "7.b.11_Fivetran": "Fivetran_ETL",
    "6.b.12_Google Dataflow": "Google_Dataflow_6b",
    "7.b.12_Google Dataflow": "Google_Dataflow_ETL",
    "6.b.13_Oracle Data Integrator": "Oracle_Data_Integrator_6b",
    "7.b.13_Oracle Data Integrator": "Oracle_Data_Integrator_ETL",
    "6.b.14_IBM DataStage": "IBM_DataStage_6b",
    "7.b.14_IBM DataStage": "IBM_DataStage_ETL",
    "6.b.15_SAP BW ETL": "SAP_BW_ETL_6b",
    "7.b.15_SAP BW ETL": "SAP_BW_ETL_ETL",
    "6.b.16_SQL Server Integration Services (SSIS)": "SQL_Server_Integration_Services_SSIS_6b",
    "7.b.16_SQL Server Integration Services (SSIS)": "SQL_Server_Integration_Services_SSIS_ETL",
    "6.b.17_SAS Data Integration": "SAS_Data_Integration_6b",
    "7.b.17_SAS Data Integration": "SAS_Data_Integration_ETL",
    "6.b.18_Qlik Sense": "Qlik_Sense_6b",
    "7.b.18_Qlik Sense": "Qlik_Sense_ETL",
    "6.b.19_Knime": "Knime_6b",
    "7.b.19_Knime": "Knime_ETL",
    "6.b.21_Não utilizo ferramentas de ETL": "Nao_utilizo_ferramentas_de_ETL_6b",
    "7.b.21_Não utilizo ferramentas de ETL": "Nao_utilizo_ferramentas_de_ETL_ETL",
    "6.c_possui_data_lake": "Sua_organizacao_possui_um_Data_Lake",
    "6.d_tecnologia_data_lake": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Lake",
    "6.e_possui_data_warehouse": "Sua_organizacao_possui_um_Data_Warehouse",
    "6.f_tecnologia_data_warehouse": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Warehouse",
    "6.g_ferramentas_de_qualidade_de_dados_(dia_a_dia)": "Quais_as_ferramentas_de_gestao_de_Qualidade_de_dados_Metadados_e_catalogo_de_dados_voce_utiliza_no_trabalho",
    "6.h_maior_tempo_gasto_como_de": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo",
    "6.h.1_Desenvolvendo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "Desenvolvendo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "6.h.2_Realizando construções de ETL\\s em ferramentas como Pentaho, Talend, Dataflow etc.": "Realizando_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "6.h.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_6h",
    "7.d.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_7d",
    "6.h.4_Atuando na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "Atuando_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "6.h.5_Modelando soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "Modelando_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "6.h.6_Desenvolvendo/cuidando da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "Desenvolvendo_cuidando_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "6.h.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "6.h.8_Cuidando da qualidade dos dados, metadados e dicionário de dados.": "Cuidando_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "7.a_rotina_como_da": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_analise_de_dados",
    "7.a.1_Processo e analiso dados utilizando linguagens de programação como Python, R etc.": "Processo_e_analiso_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "7.a.2_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_7a",
    "8.a.7_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_8a",
    "7.a.4_Utilizo API\\s para extrair dados e complementar minhas análises.": "Utilizo_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "7.a.5_Realizo experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "Realizo_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "7.a.6_Desenvolvo/cuido da manutenção de ETL\\s utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "Desenvolvo_cuido_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "7.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "7.a.8_Desenvolvo/cuido da manutenção de planilhas para atender as áreas de negócio.": "Desenvolvo_cuido_da_manutencao_de_planilhas_para_atender_as_areas_de_negocio",
    "7.a.9_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "Utilizo_ferramentas_avancadas_de_estatistica_como_SASS_PSS_Stata_etc",
    "7.b_ferramentas_etl_da": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Analyst_ETL",
    "7.c_ferramentas_autonomia_area_de_negocios": "Sua_empresa_utiliza_alguma_das_ferramentas_listadas_para_dar_mais_autonomia_em_analise_de_dados_para_as_areas_de_negocio",
    "7.c.1_Ferramentas de AutoML como H2O.ai, Data Robot, BigML etc.": "Ferramentas_de_AutoML_como_H2O_ai_Data_Robot_BigML_etc",
    "7.c.2_Point and Click Analytics como Alteryx, Knime, Rapidminer etc.": "Point_and_Click_Analytics_como_Alteryx_Knime_Rapidminer_etc",
    "7.c.3_Product metricts & Insights como Mixpanel, Amplitude, Adobe Analytics.": "Product_metrics_Insights_como_Mixpanel_Amplitude_Adobe_Analytics",
    "7.c.4_Ferramentas de análise dentro de ferramentas de CRM como Salesforce Einstein Anaytics ou Zendesk dashboards.": "Ferramentas_de_analise_dentro_de_ferramentas_de_CRM_como_Salesforce_Einstein_Analytics_ou_Zendesk_dashboards",
    "7.c.5_Minha empresa não utiliza essas ferramentas.": "Minha_empresa_nao_utiliza_essas_ferramentas",
    "7.c.6_Não sei informar.": "Nao_sei_informar",
    "7.d_maior_tempo_gasto_como_da": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_de_trabalho",
    "7.d.1_Processando e analisando dados utilizando linguagens de programação como Python, R etc.": "Processando_e_analisando_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "7.d.2_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_7d",
    "8.d.7_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc.": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_8d",
    "7.d.4_Utilizando API's para extrair dados e complementar minhas análises.": "Utilizando_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "7.d.5_Realizando experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "Realizando_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "7.d.6_Desenvolvendo/cuidando da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "Desenvolvendo_cuidando_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "7.d.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "7.d.8_Desenvolvendo/cuidando da manutenção de planilhas para atender as áreas de negócio.": "Desenvolvendo_cuidando_da_manutencao_de_planilhas_do_Excel_ou_Google_Sheets_para_atender_as_areas_de_negocio",
    "7.d.9_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_7d",
    "8.d.8_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_8d",
    "8.a_rotina_como_ds": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_ciencia_de_dados",
    "8.a.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_8a",
    "8.d.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_8d",
    "8.a.2_Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.": "Sou_responsavel_pela_coleta_e_limpeza_dos_dados_que_uso_para_analise_e_modelagem",
    "8.a.3_Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "Sou_responsavel_por_entrar_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "8.a.4_Desenvolvo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "Desenvolvo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "8.a.5_Sou responsável por colocar modelos em produção, criar os pipelines de dados, APIs de consumo e monitoramento.": "Sou_responsavel_por_colocar_modelos_em_producao_criar_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "8.a.6_Cuido da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "Cuido_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "8.a.8_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "Utilizo_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_estatisticas_e_ajustar_modelos",
    "8.a.9_Crio e dou manutenção em ETLs, DAGs e automações de pipelines de dados.": "Crio_e_dou_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "8.a.10_Crio e gerencio soluções de Feature Store e cultura de MLOps.": "Crio_e_gerencio_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "8.a.11_Sou responsável por criar e manter a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "Sou_responsavel_por_criar_e_manter_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "8.a.12_Treino e aplico LLM's para solucionar problemas de negócio.": "Treino_e_aplico_LLM_s_para_solucionar_problemas_de_negocio",
    "8.b_tecnicas_e_metodos_ds": "Quais_as_tecnicas_e_metodos_listados_abaixo_voce_costuma_utilizar_no_trabalho",
    "8.b.1_Utilizo modelos de regressão (linear, logística, GLM).": "Utilizo_modelos_de_regressao_linear_logistica_GLM",
    "8.b.2_Utilizo redes neurais ou modelos baseados em árvore para criar modelos de classificação.": "Utilizo_redes_neurais_ou_modelos_baseados_em_arvore_para_criar_modelos_de_classificacao",
    "8.b.3_Desenvolvo sistemas de recomendação (RecSys).": "Desenvolvo_sistemas_de_recomendacao_RecSys",
    "8.b.4_Utilizo métodos estatísticos Bayesianos para analisar dados.": "Utilizo_metodos_estatisticos_Bayesianos_para_analisar_dados",
    "8.b.5_Utilizo técnicas de NLP (Natural Language Processing) para análisar dados não-estruturados.": "Utilizo_tecnicas_de_NLP_Natural_Language_Processing_para_analisar_dados_nao_estruturados",
    "8.b.6_Utilizo métodos estatísticos clássicos (Testes de hipótese, análise multivariada, sobrevivência, dados longitudinais, inferência estatistica) para analisar dados.": "Utilizo_metodos_estatisticos_classicos_Testes_de_hipotese_analise_multivariada_sobrevivencia_dados_longitudinais_inferencia_estatistica_para_analisar_dados",
    "8.b.7_Utilizo cadeias de Markov ou HMM\\s para realizar análises de dados.": "Utilizo_cadeias_de_Markov_ou_HMM_s_para_realizar_analises_de_dados",
    "8.b.8_Desenvolvo técnicas de Clusterização (K-means, Spectral, DBScan etc).": "Desenvolvo_tecnicas_de_Clusterizacao_K_means_Spectral_DBScan_etc",
    "8.b.9_Realizo previsões através de modelos de Séries Temporais (Time Series).": "Realizo_previsoes_atraves_de_modelos_de_Series_Temporais_Time_Series",
    "8.b.10_Utilizo modelos de Reinforcement Learning (aprendizado por reforço).": "Utilizo_modelos_de_Reinforcement_Learning_aprendizado_por_reforco",
    "8.b.11_Utilizo modelos de Machine Learning para detecção de fraude.": "Utilizo_modelos_de_Machine_Learning_para_deteccao_de_fraude",
    "8.b.12_Utilizo métodos de Visão Computacional.": "Utilizo_metodos_de_Visao_Computacional",
    "8.b.13_Utilizo modelos de Detecção de Churn.": "Utilizo_modelos_de_Deteccao_de_Churn",
    "8.b.14_Utilizo LLM's para solucionar problemas de negócio.": "Utilizo_LLM_s_para_solucionar_problemas_de_negocio",
    "8.c_tecnologias_ds": "Quais_dessas_tecnologias_fazem_parte_do_seu_dia_a_dia_como_cientista_de_dados",
    "8.c.1_Ferramentas de BI (PowerBI, Looker, Tableau, Qlik etc).": "Ferramentas_de_BI_PowerBI_Looker_Tableau_Qlik_etc",
    "8.c.2_Planilhas (Excel, Google Sheets etc).": "Planilhas_Excel_Google_Sheets_etc",
    "8.c.3_Ambientes de desenvolvimento local (R-studio, JupyterLab, Anaconda).": "Ambientes_de_desenvolvimento_local_R_studio_JupyterLab_Anaconda",
    "8.c.4_Ambientes de desenvolvimento na nuvem (Google Colab, AWS Sagemaker, Kaggle Notebooks etc).": "Ambientes_de_desenvolvimento_na_nuvem_Google_Colab_AWS_Sagemaker_Kaggle_Notebooks_etc",
    "8.c.5_Ferramentas de AutoML (Datarobot, H2O, Auto-Keras etc).": "Ferramentas_de_AutoML_Datarobot_H2O_Auto_Keras_etc",
    "8.c.6_Ferramentas de ETL (Apache Airflow, NiFi, Stitch, Fivetran, Pentaho etc).": "Ferramentas_de_ETL_Apache_Airflow_NiFi_Stitch_Fivetran_Pentaho_etc",
    "8.c.7_Plataformas de Machine Learning (TensorFlow, Azure Machine Learning, Kubeflow etc).": "Plataformas_de_Machine_Learning_TensorFlow_Azure_Machine_Learning_Kubeflow_etc",
    "8.c.8_Feature Store (Feast, Hopsworks, AWS Feature Store, Databricks Feature Store etc).": "Feature_Store_Feast_Hopsworks_AWS_Feature_Store_Databricks_Feature_Store_etc",
    "8.c.9_Sistemas de controle de versão (Github, DVC, Neptune, Gitlab etc).": "Sistemas_de_controle_de_versao_Github_DVC_Neptune_Gitlab_etc",
    "8.c.10_Plataformas de Data Apps (Streamlit, Shiny, Plotly Dash etc).": "Plataformas_de_Data_Apps_Streamlit_Shiny_Plotly_Dash_etc",
    "8.c.11_Ferramentas de estatística avançada como SPSS, SAS, etc.": "Ferramentas_de_estatistica_avancada_como_SPSS_SAS_etc",
    "8.d_maior_tempo_gasto_como_ds": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_no_trabalho",
    "8.d.2_Coletando e limpando dos dados que uso para análise e modelagem.": "Coletando_e_limpando_os_dados_que_uso_para_analise_e_modelagem",
    "8.d.3_Entrando em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "Entrando_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "8.d.4_Desenvolvendo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "Desenvolvendo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "8.d.5_Colocando modelos em produção, criando os pipelines de dados, APIs de consumo e monitoramento.": "Colocando_modelos_em_producao_criando_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "8.d.6_Cuidando da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "Cuidando_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "8.d.9_Criando e dando manutenção em ETLs, DAGs e automações de pipelines de dados.": "Criando_e_dando_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "8.d.10_Criando e gerenciando soluções de Feature Store e cultura de MLOps.": "Criando_e_gerenciando_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "8.d.11_Criando e mantendo a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "Criando_e_mantendo_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "8.d.12_Treinando e aplicando LLM's para solucionar problemas de negócio.": "Treinando_e_aplicando_LLM_s_para_solucionar_problemas_de_negocio",
    "1.i.1_uf_onde_mora": "uf_onde_mora",
}

In [ ]:
mapa_2025 = {
    "0.a_token": "id",
    "0.d_data/hora_envio": "data_hora_envio",
    "1.a_idade": "Idade",
    "1.a.1_faixa_idade": "Faixa_idade",
    "1.b_genero": "Genero",
    "1.c_cor/raca/etnia": "Cor_raca_etnia",
    "1.d_pcd": "PCD",
    "1.e_experiencia_profissional_prejudicada": "experiencia_profissional_prejudicada",
    "1.e.1_Não acredito que minha experiência profissional seja afetada": "Nao_acredito_que_minha_experiencia_profissional_seja_afetada",
    "1.e.2_Sim, devido a minha Cor/Raça/Etnia": "Experiencia_prejudicada_devido_a_minha_Cor_Raca_Etnia",
    "1.e.3_Sim, devido a minha identidade de gênero": "Experiencia_prejudicada_devido_a_minha_identidade_de_genero",
    "1.e.4_Sim, devido ao fato de ser PCD": "Experiencia_prejudicada_devido_ao_fato_de_ser_PCD",
    "1.f_aspectos_prejudicados": "aspectos_prejudicados",
    "1.f.1_Quantidade de oportunidades de emprego/vagas recebidas": "Quantidade_de_oportunidades_de_emprego_vagas_recebidas",
    "1.f.2_Senioridade das vagas recebidas em relação à sua experiência": "Senioridade_das_vagas_recebidas_em_relacao_a_sua_experiencia",
    "1.f.3_Aprovação em processos seletivos/entrevistas": "Aprovacao_em_processos_seletivos_entrevistas",
    "1.f.4_Oportunidades de progressão de carreira": "Oportunidades_de_progressao_de_carreira",
    "1.f.5_Velocidade de progressão de carreira": "Velocidade_de_progressao_de_carreira",
    "1.f.6_Nível de cobrança no trabalho/Stress no trabalho": "Nivel_de_cobranca_no_trabalho_Stress_no_trabalho",
    "1.f.7_Atenção dada pelas pessoas diante das minhas opiniões e ideias": "Atencao_dada_diante_das_minhas_opinioes_e_ideias",
    "1.f.8_Relação com outras pessoas da empresa, em momentos de trabalho": "Relacao_com_outros_membros_da_empresa_em_momentos_de_trabalho",
    "1.f.9_Relação com outras pessoas da empresa, em momentos de integração e outros momentos fora do trabalho": "Relacao_com_outros_membros_da_empresa_em_momentos_de_integracao_e_outros_momentos_fora_do_trabalho",
    "1.g_vive_no_brasil": "vive_no_brasil",
    "1.h_pais_onde_mora": "pais_onde_mora",
    "1.i_estado_onde_mora": "Estado_onde_mora",
    "1.i.1_uf_onde_mora": "uf_onde_mora",
    "1.i.2_regiao_onde_mora": "Regiao_onde_mora",
    "1.k_estado_de_origem": "Estado_de_origem",
    "1.k.1_uf_de_origem": "uf_de_origem",
    "1.k.2_regiao_de_origem": "Regiao_de_origem",
    "1.j_vive_no_estado_de_formacao": "vive_no_estado_de_formacao",
    "1.l_nivel_de_ensino": "Nivel_de_Ensino",
    "1.m_área_de_formação": "Area_de_Formacao",
    "2.a_situação_de_trabalho": "Qual_sua_situacao_atual_de_trabalho",
    "2.b_setor": "Setor",
    "2.c_numero_de_funcionarios": "Numero_de_Funcionarios",
    "2.d_atua_como_gestor": "Gestor",
    "2.e_cargo_como_gestor": "Cargo_como_Gestor",
    "2.f_cargo_atual": "Cargo_Atual",
    "2.g_nivel": "Nivel",
    "2.h_faixa_salarial": "Faixa_salarial",
    "2.i_tempo_de_experiencia_em_dados": "Quanto_tempo_de_experiencia_na_area_de_dados_voce_tem",
    "2.j_tempo_de_experiencia_em_ti": "Quanto_tempo_de_experiencia_na_area_de_TI_Engenharia_de_Software_voce_teve_antes_de_comecar_a_trabalhar_na_area_de_dados",
    "2.k_satisfeito_atualmente": "Voce_esta_satisfeito_na_sua_empresa_atual",
    "2.l_motivo_insatisfacao": "Qual_o_principal_motivo_da_sua_insatisfacao_com_a_empresa_atual",
    "2.l.1_Remuneração/Salário": "Salario_atual_nao_corresponde_ao_mercado",
    "2.l.2_Benefícios": "Gostaria_de_receber_mais_beneficios",
    "2.l.3_Propósito do trabalho e da empresa": "Proposito_do_trabalho_e_da_empresa_2l",
    "2.o.3_Propósito do trabalho e da empresa": "Proposito_do_trabalho_e_da_empresa_2o",
    "2.l.4_Flexibilidade de trabalho remoto": "Flexibilidade_de_trabalho_remoto_2l",
    "2.o.4_Flexibilidade de trabalho remoto": "Flexibilidade_de_trabalho_remoto_2o",
    "2.l.5_Ambiente e clima de trabalho": "O_clima_de_trabalho_ambiente_nao_e_bom",
    "2.l.6_Oportunidade de aprendizado e trabalhar com referências": "Oportunidade_de_aprendizado_e_trabalhar_com_referencias",
    "2.l.7_Oportunidades de crescimento": "Falta_de_oportunidade_de_crescimento_no_emprego_atual",
    "2.l.8_Maturidade da empresa em termos de tecnologia e dados": "Falta_de_maturidade_analitica_na_empresa",
    "2.l.9_Relação com os gestores e líderes": "Nao_tenho_uma_boa_relacao_com_meu_lider_gestor",
    "2.l.10_Reputação que a empresa tem no mercado": "Reputacao_que_a_empresa_tem_no_mercado_2l",
    "2.o.10_Reputação que a empresa tem no mercado": "Reputacao_que_a_empresa_tem_no_mercado_2o",
    "2.l.11_Gostaria de trabalhar em outra área": "Gostaria_de_trabalhar_em_em_outra_area_de_atuacao",
    "2.m_participou_de_entrevistas_ultimos_6m": "Voce_participou_de_entrevistas_de_emprego_nos_ultimos_6_meses",
    "2.n_planos_de_mudar_de_emprego_6m": "Voce_pretende_mudar_de_emprego_nos_proximos_6_meses",
    "2.o_criterios_para_escolha_de_emprego": "Quais_os_principais_criterios_que_voce_leva_em_consideracao_no_momento_de_decidir_onde_trabalhar",
    "2.o.1_Remuneração/Salário": "Remuneracao_Salario",
    "2.o.2_Benefícios": "Beneficios",
    "2.o.5_Ambiente e clima de trabalho": "Ambiente_e_clima_de_trabalho",
    "2.o.6_Oportunidade de aprendizado e trabalhar com referências": "Oportunidade_de_aprendizado_e_trabalhar_com_referencias_na_area",
    "2.o.7_Plano de carreira e oportunidades de crescimento": "Plano_de_carreira_e_oportunidades_de_crescimento_profissional",
    "2.o.8_Maturidade da empresa em termos de tecnologia e dados": "Maturidade_da_empresa_em_termos_de_tecnologia_e_dados",
    "2.o.9_Qualidade dos gestores e líderes": "Qualidade_dos_gestores_e_lideres",
    "2.p_empresa_passou_por_layoff_em_2025": "Empresa_que_trabaha_passou_por_layoff_em_2023",
    "2.q_modelo_de_trabalho_atual": "Atualmente_qual_a_sua_forma_de_trabalho",
    "2.r_modelo_de_trabalho_ideal": "Qual_a_forma_de_trabalho_ideal_para_voce",
    "2.s_atitude_em_caso_de_retorno_presencial": "Caso_sua_empresa_decida_pelo_modelo_100_presencial_qual_sera_sua_atitude",
    "3.a_numero_de_pessoas_em_dados": "Qual_o_numero_aproximado_de_pessoas_que_atuam_com_dados_na_sua_empresa_hoje",
    "3.b_cargos_no_time_de_dados_da_empresa": "Quais_desses_papeis_cargos_fazem_parte_do_time_ou_chapter_de_dados_da_sua_empresa",
    "3.b.1_Analytics Engineer": "Analytics_Engineer",
    "3.b.2_Engenharia de Dados/Data Engineer": "Engenharia_de_Dados_Data_Engineer",
    "3.b.3_Analista de Dados/Data Analyst": "Analista_de_Dados_Data_Analyst",
    "3.b.4_Cientista de Dados/Data Scientist": "Cientista_de_Dados_Data_Scientist",
    "3.b.5_Database Administrator/DBA": "Database_Administrator_DBA",
    "3.b.6_Analista de Business Intelligence/BI": "Analista_de_Business_Intelligence_BI",
    "3.b.7_Arquiteto de Dados/Data Architect": "Arquiteto_de_Dados_Data_Architect",
    "3.b.8_Data Product Manager/DPM": "Data_Product_Manager_DPM",
    "3.b.9_Business Analyst": "Business_Analyst",
    "3.b.10_ML Engineer/AI Engineer": "ML_Engineer_AI_Engineer",
    "3.c_responsabilidades_como_gestor": "Quais_dessas_responsabilidades_fazem_parte_da_sua_rotina_atual_de_trabalho_como_gestor",
    "3.c.1_Pensar na visão de longo prazo de dados": "Pensar_na_visao_de_longo_prazo_de_dados_da_empresa_e_fortalecimento_da_cultura_analitica_da_companhia",
    "3.c.2_Organização de treinamentos e iniciativas": "Organizacao_de_treinamentos_e_iniciativas_com_o_objetivo_de_aumentar_a_maturidade_analitica_das_areas_de_negocios",
    "3.c.3_Atração, seleção e contratação": "Atracao_selecao_e_contratacao_de_talentos_para_o_time_de_dados",
    "3.c.4_Decisão sobre contratação de ferramentas": "Decisao_sobre_contratacao_de_ferramentas_e_tecnologias_relacionadas_a_dados",
    "3.c.5_gestor da equipe de engenharia de dados": "Sou_gestor_da_equipe_responsavel_pela_engenharia_de_dados_e_por_manter_o_Data_Lake_da_empresa_como_fonte_unica_dos_dados_garantindo_a_qualidade_e_confiabilidade_da_informacao",
    "3.c.6_gestor da equipe de estudos, relatórios": "Sou_gestor_da_equipe_responsavel_pela_entrega_de_dados_estudos_relatorios_e_dashboards_para_as_areas_de_negocio_da_empresa",
    "3.c.7_gestor da equipe de Inteligência Artificial e Machine Learning": "Sou_gestor_da_equipe_responsavel_por_iniciativas_e_projetos_envolvendo_Inteligencia_Artificial_e_Machine_Learning",
    "3.c.8_Apesar de ser gestor ainda atuo na parte técnica": "Apesar_de_ser_gestor_ainda_atuo_na_parte_tecnica_construindo_solucoes_analises_modelos_etc",
    "3.c.9_Gestão de projetos de dados": "Gestao_de_projetos_de_dados_cuidando_das_etapas_equipes_envolvidas_atingimento_dos_objetivos_etc",
    "3.c.10_Gestão de produtos de dados": "Gestao_de_produtos_de_dados_cuidando_da_visao_dos_produtos_backlog_feedback_de_usuarios_etc",
    "3.c.11_Gestão de pessoas": "Gestao_de_pessoas_apoio_no_desenvolvimento_das_pessoas_evolucao_de_carreira",
    "3.d_desafios_como_gestor": "Quais_sao_os_3_maiores_desafios_que_voce_tem_como_gestor_no_atual_momento",
    "3.d.1_Contratar talentos": "a_Contratar_novos_talentos",
    "3.d.2_Reter talentos": "b_Reter_talentos",
    "3.d.3_Convencer a empresa a aumentar investimentos": "c_Convencer_a_empresa_a_aumentar_os_investimentos_na_area_de_dados",
    "3.d.4_Gestão de equipes no ambiente remoto": "d_Gestao_de_equipes_no_ambiente_remoto",
    "3.d.5_Gestão de projetos envolvendo áreas multidisciplinares": "e_Gestao_de_projetos_envolvendo_areas_multidisciplinares_da_empresa",
    "3.d.6_Organizar as informações com qualidade e confiabilidade": "f_Organizar_as_informacoes_e_garantir_a_qualidade_e_confiabilidade",
    "3.d.7_Processar e armazenar um alto volume de dados": "g_Conseguir_processar_e_armazenar_um_alto_volume_de_dados",
    "3.d.8_Gerar valor para as áreas de negócios": "h_Conseguir_gerar_valor_para_as_areas_de_negocios_atraves_de_estudos_e_experimentos",
    "3.d.9_Desenvolver e manter modelos Machine Learning em produção": "i_Desenvolver_e_manter_modelos_Machine_Learning_em_producao",
    "3.d.10_Gerenciar a expectativa das áreas": "j_Gerenciar_a_expectativa_das_areas_de_negocio_em_relacao_as_entregas_das_equipes_de_dados",
    "3.d.11_Garantir a manutenção dos projetos e modelos em produção": "k_Garantir_a_manutencao_dos_projetos_e_modelos_em_producao_em_meio_ao_crescimento_da_empresa",
    "3.d.12_Conseguir levar inovação para a empresa": "Conseguir_levar_inovacao_para_a_empresa_atraves_dos_dados",
    "3.d.13_Garantir (ROI) em projetos de dados": "Garantir_retorno_do_investimento_ROI_em_projetos_de_dados",
    "3.d.14_Dividir o tempo entre entregas técnicas e gestão": "Dividir_o_tempo_entre_entregas_tecnicas_e_gestao",
    "3.e_ai_generativa_e_llm_é_uma_prioridade?": "AI_Generativa_e_uma_prioridade_em_sua_empresa",
    "3.f_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "Tipos_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "3.f.1 Colaboradores usando AI generativa de forma independente e descentralizada": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_3f",
    "4.i.1 Colaboradores usando AI generativa de forma independente e descentralizada": "Colaboradores_usando_AI_generativa_de_forma_independente_e_descentralizada_4i",
    "3.f.2 Direcionamento centralizado do uso de AI generativa": "Direcionamento_centralizado_do_uso_de_AI_generativa_3f",
    "4.i.2 Direcionamento centralizado do uso de AI generativa": "Direcionamento_centralizado_do_uso_de_AI_generativa_4i",
    "3.f.3 Desenvolvedores utilizando Copilots": "Desenvolvedores_utilizando_Copilots_3f",
    "4.i.3 Desenvolvedores utilizando Copilots": "Desenvolvedores_utilizando_Copilots_4i",
    "3.f.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos",
    "3.f.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_3f",
    "4.i.5 AI Generativa e LLMs para melhorar produtos internos para os colaboradores": "AI_Generativa_e_LLMs_para_melhorar_produtos_internos_para_os_colaboradores_4i",
    "3.f.6 IA Generativa e LLMs como principal frente do negócio": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_3f",
    "4.i.6 IA Generativa e LLMs como principal frente do negócio": "IA_Generativa_e_LLMs_como_principal_frente_do_negocio_4i",
    "3.f.7 IA Generativa e LLMs não é prioridade": "IA_Generativa_e_LLMs_nao_e_prioridade_3f",
    "4.i.7 IA Generativa e LLMs não é prioridade": "IA_Generativa_e_LLMs_nao_e_prioridade_4i",
    "3.f.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_3f",
    "4.i.8 Não sei opinar sobre o uso de IA Generativa e LLMs na empresa": "Nao_sei_opinar_sobre_o_uso_de_IA_Generativa_e_LLMs_na_empresa_4i",
    "3.g_empresa_está_conseguindo_ter_bons_resultados_com_llms": "presa_esta_conseguindo_ter_bons_resultados_com_llms",
    "3.h_motivos_para_não_usar_ai_generativa_e_llm": "Motivos_que_levam_a_empresa_a_nao_usar_AI_Genrativa_e_LLMs",
    "3.h.1 Falta de compreensão dos casos de uso": "Falta_de_compreensao_dos_casos_de_uso",
    "3.h.2 Falta de confiabilidade das saídas (alucinação dos modelos)": "Falta_de_confiabilidade_das_saidas_alucinacao_dos_modelos",
    "3.h.3 Incerteza em relação a regulamentação": "Incerteza_em_relacao_a_regulamentacao",
    "3.h.4 Preocupações com segurança e privacidade de dados": "Preocupacoes_com_seguranca_e_privacidade_de_dados",
    "3.h.5 Retorno sobre investimento (ROI) não comprovado de IA Generativa": "Retorno_sobre_investimento_ROI_nao_comprovado_de_IA_Generativa",
    "3.h.6 Dados da empresa não estão prontos para uso de IA Generativa": "Dados_da_empresa_nao_estao_prontos_para_uso_de_IA_Generativa",
    "3.h.7 Falta de expertise ou falta de recursos": "Falta_de_expertise_ou_falta_de_recursos",
    "3.h.8 Alta direção da empresa não vê valor ou não vê como prioridade": "Alta_direcao_da_empresa_nao_ve_valor_ou_nao_ve_como_prioridade",
    "3.h.9 Preocupações com propriedade intelectual": "Preocupacoes_com_propriedade_intelectual",
    "4.a_funcao_de_atuacao": "Mesmo_que_esse_nao_seja_seu_cargo_formal_voce_considera_que_sua_atuacao_no_dia_a_dia_reflete_alguma_das_opcoes_listadas_abaixo",
    "4.a.1_atuacao_em_dados": "Atuacao",
    "4.b_fontes_de_dados_(dia_a_dia)": "Quais_das_fontes_de_dados_listadas_voce_ja_analisou_ou_processou_no_trabalho",
    "4.b.1_Dados relacionais (estruturados em bancos SQL)": "Dados_relacionais_estruturados_em_bancos_SQL",
    "4.b.2_Dados armazenados em bancos NoSQL": "Dados_armazenados_em_bancos_NoSQL",
    "4.b.3_Imagens": "Imagens",
    "4.b.4_Textos/Documentos": "Textos_Documentos",
    "4.b.5_Vídeos": "Videos",
    "4.b.6_Áudios": "Audios",
    "4.b.7_Planilhas": "Planilhas",
    "4.b.8_Dados georeferenciados": "Dados_georeferenciados",
    "4.c_linguagem_preferida": "Quais_das_linguagens_listadas_abaixo_voce_utiliza_no_trabalho_Linguagem",
    "4.c.1_SQL": "SQL_Linguagem",
    "4.c.2_R": "R_Linguagem",
    "4.c.3_Python": "Python_Linguagem",
    "4.c.4_C/C++/C#": "C_C_C_Linguagem",
    "4.c.5_Julia": "Julia_Linguagem",
    "4.c.6_Visual Basic/VBA": "Visual_Basic_VBA_Linguagem",
    "4.c.7_Scala": "Scala_Linguagem",
    "4.c.8_DAX": "DAX_Linguagem",
    "4.c.9_Rust": "Rust_Linguagem",
    "4.c.10_Não utilizo nenhuma das linguagens listadas": "Nao_utilizo_nenhuma_linguagem_Linguagem",
    "4.d_banco_de_dados_(dia_a_dia)": "Quais_dos_bancos_de_dados_fontes_de_dados_listados_abaixo_voce_utiliza_no_trabalho_BD",
    "4.d.1_MySQL": "MySQL_BD",
    "4.d.2_Oracle": "Oracle_BD",
    "4.d.3_SQL SERVER": "SQL_SERVER_BD",
    "4.d.4_Amazon Aurora ou RDS": "Amazon_Aurora_ou_RDS_BD",
    "4.d.5_DynamoDB": "DynamoDB_BD",
    "4.d.6_CoachDB": "CoachDB_BD",
    "4.d.7_Cassandra": "Cassandra_BD",
    "4.d.8_MongoDB": "MongoDB_BD",
    "4.d.9_MariaDB": "MariaDB_BD",
    "4.d.10_Datomic": "Datomic_BD",
    "4.d.11_S3": "S3_BD",
    "4.d.12_PostgreSQL": "PostgreSQL_BD",
    "4.d.13_ElasticSearch": "ElasticSearch_BD",
    "4.d.14_DB2": "DB2_BD",
    "4.d.15_Microsoft Access": "Microsoft_Access_BD",
    "4.d.16_SQLite": "SQLite_BD",
    "4.d.17_Sybase": "Sybase_BD",
    "4.d.18_Firebase": "Firebase_BD",
    "4.d.19_Vertica": "Vertica_BD",
    "4.d.20_Redis": "Redis_BD",
    "4.d.21_Neo4J": "Neo4J_BD",
    "4.d.22_Google BigQuery": "Google_BigQuery_BD",
    "4.d.23_Google Firestore": "Google_Firestore_BD",
    "4.d.24_Amazon Redshift": "Amazon_Redshift_BD",
    "4.d.25_Amazon Athena": "Amazon_Athena_BD",
    "4.d.26_Snowflake": "Snowflake_BD",
    "4.d.27_Databricks": "Databricks_BD",
    "6.b.20_Databricks": "Databricks_6b",
    "7.b.20_Databricks": "Databricks_ETL",
    "4.d.28_HBase": "HBase_BD",
    "4.d.29_Presto": "Presto_BD",
    "4.d.30_Splunk": "Splunk_BD",
    "4.d.31_SAP HANA": "SAP_HANA_BD",
    "4.d.32_Hive": "Hive_BD",
    "4.d.33_Firebird": "Firebird_BD",
    "4.e_cloud_(dia_a_dia)": "Dentre_as_opcoes_listadas_qual_sua_Cloud_preferida_cloud",
    "4.e.1_Amazon Web Services (AWS)": "Amazon_Web_Services_AWS_cloud",
    "4.e.2_Google Cloud (GCP)": "Google_Cloud_GCP_cloud",
    "4.e.3_Azure (Microsoft)": "Azure_Microsoft_cloud",
    "4.e.4_Oracle Cloud": "Oracle_Cloud_cloud",
    "4.e.5_IBM": "IBM_cloud",
    "4.e.6_Servidores On Premise/Não utilizamos Cloud": "Servidores_On_Premise_Nao_utilizamos_Cloud_cloud",
    "4.e.7_Cloud Própria": "Cloud_Propria_cloud",
    "4.f_cloud_preferida": "Cloud_preferida_cloud",
    "4.g_ferramenta_de_bi_(dia_a_dia)": "Ferramenta_de_BI_utilizada_no_dia_a_dia_BI",
    "4.g.1_Microsoft PowerBI": "Microsoft_PowerBI_BI",
    "4.g.2_Qlik View/Qlik Sense": "Qlik_View_Qlik_Sense_BI",
    "4.g.3_Tableau": "Tableau_BI",
    "4.g.4_Metabase": "Metabase_BI",
    "4.g.5_Superset": "Superset_BI",
    "4.g.6_Redash": "Redash_BI",
    "4.g.7_Looker": "Looker_BI",
    "4.g.8_Looker Studio(Google Data Studio)": "Looker_Studio_Google_Data_Studio_BI",
    "4.g.9_Amazon Quicksight": "Amazon_Quicksight_BI",
    "4.g.10_Alteryx": "Alteryx_BI",
    "6.b.9_Alteryx": "Alteryx_6b",
    "7.b.9_Alteryx": "Alteryx_ETL",
    "4.g.11_SAP Business Objects/SAP Analytics": "SAP_Business_Objects_SAP_Analytics_BI",
    "4.g.12_Oracle Business Intelligence": "Oracle_Business_Intelligence_BI",
    "4.g.13_Salesforce/Einstein Analytics": "Salesforce_Einstein_Analytics_BI",
    "4.g.14_SAS Visual Analytics": "SAS_Visual_Analytics_BI",
    "4.g.15_Grafana": "Grafana_BI",
    "4.g.16_Pentaho": "Pentaho_BI",
    "6.b.8_Pentaho": "Pentaho_6b",
    "7.b.8_Pentaho": "Pentaho_ETL",
    "4.g.17_Fazemos todas as análises utilizando apenas Excel ou planilhas do google": "Fazemos_todas_as_analises_utilizando_apenas_Excel_ou_planilhas_do_google_BI",
    "4.g.18_Não utilizo nenhuma ferramenta de BI no trabalho": "Nao_utilizo_nenhuma_ferramenta_de_BI_no_trabalho_BI",
    "4.h_ferramenta_de_bi_preferida": "Qual_sua_ferramenta_de_BI_preferida",
    "4.i_tipo_de_uso_de_ai_generativa_e_llm_na_empresa": "Qual_o_tipo_de_uso_de_AI_Generativa_e_LLMs_na_empresa",
    "4.i.4 AI Generativa e LLMs para melhorar produtos externos para os clientes finais": "AI_Generativa_e_LLMs_para_melhorar_produtos_externos_para_os_clientes_finais",
    "4.j_usa_chatgpt_ou_copilot_no_trabalho?": "Utiliza_ChatGPT_ou_LLMs_no_trabalho",
    "4.j.1 Não uso soluções de AI Generativa com foco em produtividade": "Nao_uso_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.j.2 Uso soluções gratuitas de AI Generativa com foco em produtividade": "Uso_solucoes_gratuitas_de_AI_Generativa_com_foco_em_produtividade",
    "4.j.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade": "Uso_e_pago_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.j.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade": "A_empresa_que_trabalho_paga_pelas_solucoes_de_AI_Generativa_com_foco_em_produtividade",
    "4.j.5 Uso soluções do tipo Copilot": "Uso_solucoes_do_tipo_Copilot",
    "5.a_objetivo_na_area_de_dados": "Qual_seu_objetivo_na_area_de_dados",
    "5.b_oportunidade_buscada": "Qual_oportunidade_voce_esta_buscando",
    "5.c_tempo_em_busca_de_oportunidade": "Ha_quanto_tempo_voce_busca_uma_oportunidade_na_area_de_dados",
    "5.d_experiencia_em_processos_seletivos": "Como_tem_sido_a_busca_por_um_emprego_na_area_de_dados",
    "6.a_rotina_como_de": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_como_engenheiro_de_dados",
    "6.a.1_Desenvolvo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "Desenvolvo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "6.a.2_Realizo construções de ETL's em ferramentas como Pentaho, Talend, Dataflow etc.": "Realizo_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "6.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_6a",
    "7.a.3_Crio consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Crio_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_7a",
    "6.a.4_Atuo na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "Atuo_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "6.a.5_Modelo soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "Modelo_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "6.a.6_Desenvolvo/cuido da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "Desenvolvo_cuido_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "6.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "6.a.8_Cuido da qualidade dos dados, metadados e dicionário de dados.": "Cuido_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "6.a.9_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_6a",
    "6.h.9_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_6h",
    "7.a.10_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_7a",
    "7.d.10_Nenhuma das opções listadas refletem meu dia a dia.": "Nenhuma_das_opcoes_listadas_refletem_meu_dia_a_dia_7d",
    "6.b_ferramentas_etl_de": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Engineer",
    "6.b.1_Scripts Python": "Scripts_Python_6b",
    "7.b.1_Scripts Python": "Scripts_Python_ETL",
    "6.b.2_SQL & Stored Procedures": "SQL_Stored_Procedures_6b",
    "7.b.2_SQL & Stored Procedures": "SQL_Stored_Procedures_ETL",
    "6.b.3_Apache Airflow": "Apache_Airflow_6b",
    "7.b.3_Apache Airflow": "Apache_Airflow_ETL",
    "6.b.4_Apache NiFi": "Apache_NiFi_6b",
    "7.b.4_Apache NiFi": "Apache_NiFi_ETL",
    "6.b.5_Luigi": "Luigi_6b",
    "7.b.5_Luigi": "Luigi_ETL",
    "6.b.6_AWS Glue": "AWS_Glue_6b",
    "7.b.6_AWS Glue": "AWS_Glue_ETL",
    "6.b.7_Talend": "Talend_6b",
    "7.b.7_Talend": "Talend_ETL",
    "6.b.10_Stitch": "Stitch_6b",
    "7.b.10_Stitch": "Stitch_ETL",
    "6.b.11_Fivetran": "Fivetran_6b",
    "7.b.11_Fivetran": "Fivetran_ETL",
    "6.b.12_Google Dataflow": "Google_Dataflow_6b",
    "7.b.12_Google Dataflow": "Google_Dataflow_ETL",
    "6.b.13_Oracle Data Integrator": "Oracle_Data_Integrator_6b",
    "7.b.13_Oracle Data Integrator": "Oracle_Data_Integrator_ETL",
    "6.b.14_IBM DataStage": "IBM_DataStage_6b",
    "7.b.14_IBM DataStage": "IBM_DataStage_ETL",
    "6.b.15_SAP BW ETL": "SAP_BW_ETL_6b",
    "7.b.15_SAP BW ETL": "SAP_BW_ETL_ETL",
    "6.b.16_SQL Server Integration Services (SSIS)": "SQL_Server_Integration_Services_SSIS_6b",
    "7.b.16_SQL Server Integration Services (SSIS)": "SQL_Server_Integration_Services_SSIS_ETL",
    "6.b.17_SAS Data Integration": "SAS_Data_Integration_6b",
    "7.b.17_SAS Data Integration": "SAS_Data_Integration_ETL",
    "6.b.18_Qlik Sense": "Qlik_Sense_6b",
    "7.b.18_Qlik Sense": "Qlik_Sense_ETL",
    "6.b.19_Knime": "Knime_6b",
    "7.b.19_Knime": "Knime_ETL",
    "6.b.21_Não utilizo ferramentas de ETL": "Nao_utilizo_ferramentas_de_ETL_6b",
    "7.b.21_Não utilizo ferramentas de ETL": "Nao_utilizo_ferramentas_de_ETL_ETL",
    "6.c_possui_data_lake": "Sua_organizacao_possui_um_Data_Lake",
    "6.d_tecnologia_data_lake": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Lake",
    "6.e_possui_data_warehouse": "Sua_organizacao_possui_um_Data_Warehouse",
    "6.f_tecnologia_data_warehouse": "Qual_tecnologia_utilizada_como_plataforma_do_Data_Warehouse",
    "6.g_ferramentas_de_qualidade_de_dados_(dia_a_dia)": "Quais_as_ferramentas_de_gestao_de_Qualidade_de_dados_Metadados_e_catalogo_de_dados_voce_utiliza_no_trabalho",
    "6.h_maior_tempo_gasto_como_de": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo",
    "6.h.1_Desenvolvendo pipelines de dados utilizando linguagens de programação como Python, Scala, Java etc.": "Desenvolvendo_pipelines_de_dados_utilizando_linguagens_de_programacao_como_Python_Scala_Java_etc",
    "6.h.2_Realizando construções de ETL\\s em ferramentas como Pentaho, Talend, Dataflow etc.": "Realizando_construcoes_de_ETL_s_em_ferramentas_como_Pentaho_Talend_Dataflow_etc",
    "6.h.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_6h",
    "7.d.3_Criando consultas através da linguagem SQL para exportar informações e compartilhar com as áreas de negócio.": "Criando_consultas_atraves_da_linguagem_SQL_para_exportar_informacoes_e_compartilhar_com_as_areas_de_negocio_7d",
    "6.h.4_Atuando na integração de diferentes fontes de dados através de plataformas proprietárias como Stitch Data, Fivetran etc.": "Atuando_na_integracao_de_diferentes_fontes_de_dados_atraves_de_plataformas_proprietarias_como_Stitch_Data_Fivetran_etc",
    "6.h.5_Modelando soluções de arquitetura de dados, criando componentes de ingestão de dados, transformação e recuperação da informação.": "Modelando_solucoes_de_arquitetura_de_dados_criando_componentes_de_ingestao_de_dados_transformacao_e_recuperacao_da_informacao",
    "6.h.6_Desenvolvendo/cuidando da manutenção de repositórios de dados baseados em streaming de eventos como Data Lakes e Data Lakehouses.": "Desenvolvendo_cuidando_da_manutencao_de_repositorios_de_dados_baseados_em_streaming_de_eventos_como_Data_Lakes_e_Data_Lakehouses",
    "6.h.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_como_Data_Warehouses_Data_Marts_etc",
    "6.h.8_Cuidando da qualidade dos dados, metadados e dicionário de dados.": "Cuidando_da_qualidade_dos_dados_metadados_e_dicionario_de_dados",
    "7.a_rotina_como_da": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_analise_de_dados",
    "7.a.1_Processo e analiso dados utilizando linguagens de programação como Python, R etc.": "Processo_e_analiso_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "7.a.2_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_7a",
    "8.a.7_Realizo construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc": "Realizo_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_8a",
    "7.a.4_Utilizo API\\s para extrair dados e complementar minhas análises.": "Utilizo_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "7.a.5_Realizo experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "Realizo_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "7.a.6_Desenvolvo/cuido da manutenção de ETL\\s utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "Desenvolvo_cuido_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "7.a.7_Atuo na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts etc.": "Atuo_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "7.a.8_Desenvolvo/cuido da manutenção de planilhas para atender as áreas de negócio.": "Desenvolvo_cuido_da_manutencao_de_planilhas_para_atender_as_areas_de_negocio",
    "7.a.9_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "Utilizo_ferramentas_avancadas_de_estatistica_como_SASS_PSS_Stata_etc",
    "7.b_ferramentas_etl_da": "Quais_as_ferramentas_tecnologias_de_ETL_que_voce_utiliza_no_trabalho_como_Data_Analyst_ETL",
    "7.c_ferramentas_autonomia_area_de_negocios": "Sua_empresa_utiliza_alguma_das_ferramentas_listadas_para_dar_mais_autonomia_em_analise_de_dados_para_as_areas_de_negocio",
    "7.c.1_Ferramentas de AutoML como H2O.ai, Data Robot, BigML etc.": "Ferramentas_de_AutoML_como_H2O_ai_Data_Robot_BigML_etc",
    "7.c.2_Point and Click Analytics como Alteryx, Knime, Rapidminer etc.": "Point_and_Click_Analytics_como_Alteryx_Knime_Rapidminer_etc",
    "7.c.3_Product metricts & Insights como Mixpanel, Amplitude, Adobe Analytics.": "Product_metrics_Insights_como_Mixpanel_Amplitude_Adobe_Analytics",
    "7.c.4_Ferramentas de análise dentro de ferramentas de CRM como Salesforce Einstein Anaytics ou Zendesk dashboards.": "Ferramentas_de_analise_dentro_de_ferramentas_de_CRM_como_Salesforce_Einstein_Analytics_ou_Zendesk_dashboards",
    "7.c.5_Minha empresa não utiliza essas ferramentas.": "Minha_empresa_nao_utiliza_essas_ferramentas",
    "7.c.6_Não sei informar.": "Nao_sei_informar",
    "7.d_maior_tempo_gasto_como_da": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_de_trabalho",
    "7.d.1_Processando e analisando dados utilizando linguagens de programação como Python, R etc.": "Processando_e_analisando_dados_utilizando_linguagens_de_programacao_como_Python_R_etc",
    "7.d.2_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik etc.": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_7d",
    "8.d.7_Realizando construções de dashboards em ferramentas de BI como PowerBI, Tableau, Looker, Qlik, etc.": "Realizando_construcoes_de_dashboards_em_ferramentas_de_BI_como_PowerBI_Tableau_Looker_Qlik_etc_8d",
    "7.d.4_Utilizando API's para extrair dados e complementar minhas análises.": "Utilizando_API_s_para_extrair_dados_e_complementar_minhas_analises",
    "7.d.5_Realizando experimentos e estudos utilizando metodologias estatísticas como teste de hipótese, modelos de regressão etc.": "Realizando_experimentos_e_estudos_utilizando_metodologias_estatisticas_como_teste_de_hipotese_modelos_de_regressao_etc",
    "7.d.6_Desenvolvendo/cuidando da manutenção de ETL's utilizando tecnologias como Talend, Pentaho, Airflow, Dataflow etc.": "Desenvolvendo_cuidando_da_manutencao_de_ETL_s_utilizando_tecnologias_como_Talend_Pentaho_Airflow_Dataflow_etc",
    "7.d.7_Atuando na modelagem dos dados, com o objetivo de criar conjuntos de dados como Data Warehouses, Data Marts, Datasets etc.": "Atuando_na_modelagem_dos_dados_com_o_objetivo_de_criar_conjuntos_de_dados_Data_Warehouses_Data_Marts_etc",
    "7.d.8_Desenvolvendo/cuidando da manutenção de planilhas para atender as áreas de negócio.": "Desenvolvendo_cuidando_da_manutencao_de_planilhas_do_Excel_ou_Google_Sheets_para_atender_as_areas_de_negocio",
    "7.d.9_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises de dados.": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_7d",
    "8.d.8_Utilizando ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "Utilizando_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_8d",
    "8.a_rotina_como_ds": "Quais_das_opcoes_abaixo_fazem_parte_da_sua_rotina_no_trabalho_atual_com_ciencia_de_dados",
    "8.a.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_8a",
    "8.d.1_Estudos Ad-hoc com o objetivo de confirmar hipóteses, realizar modelos preditivos, forecasts, análise de cluster para resolver problemas pontuais e responder perguntas das áreas de negócio.": "Estudos_Ad_hoc_com_o_objetivo_de_confirmar_hipoteses_realizar_modelos_preditivos_forecasts_analise_de_cluster_para_resolver_problemas_pontuais_e_responder_perguntas_das_areas_de_negocio_8d",
    "8.a.2_Sou responsável pela coleta e limpeza dos dados que uso para análise e modelagem.": "Sou_responsavel_pela_coleta_e_limpeza_dos_dados_que_uso_para_analise_e_modelagem",
    "8.a.3_Sou responsável por entrar em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "Sou_responsavel_por_entrar_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "8.a.4_Desenvolvo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "Desenvolvo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "8.a.5_Sou responsável por colocar modelos em produção, criar os pipelines de dados, APIs de consumo e monitoramento.": "Sou_responsavel_por_colocar_modelos_em_producao_criar_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "8.a.6_Cuido da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "Cuido_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "8.a.8_Utilizo ferramentas avançadas de estatística como SAS, SPSS, Stata etc, para realizar análises.": "Utilizo_ferramentas_avancadas_de_estatistica_como_SAS_SPSS_Stata_etc_para_realizar_analises_estatisticas_e_ajustar_modelos",
    "8.a.9_Crio e dou manutenção em ETLs, DAGs e automações de pipelines de dados.": "Crio_e_dou_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "8.a.10_Crio e gerencio soluções de Feature Store e cultura de MLOps.": "Crio_e_gerencio_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "8.a.11_Sou responsável por criar e manter a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "Sou_responsavel_por_criar_e_manter_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "8.a.12_Treino e aplico LLM's para solucionar problemas de negócio.": "Treino_e_aplico_LLM_s_para_solucionar_problemas_de_negocio",
    "8.b_tecnicas_e_metodos_ds": "Quais_as_tecnicas_e_metodos_listados_abaixo_voce_costuma_utilizar_no_trabalho",
    "8.b.1_Utilizo modelos de regressão (linear, logística, GLM).": "Utilizo_modelos_de_regressao_linear_logistica_GLM",
    "8.b.2_Utilizo redes neurais ou modelos baseados em árvore para criar modelos de classificação.": "Utilizo_redes_neurais_ou_modelos_baseados_em_arvore_para_criar_modelos_de_classificacao",
    "8.b.3_Desenvolvo sistemas de recomendação (RecSys).": "Desenvolvo_sistemas_de_recomendacao_RecSys",
    "8.b.4_Utilizo métodos estatísticos Bayesianos para analisar dados.": "Utilizo_metodos_estatisticos_Bayesianos_para_analisar_dados",
    "8.b.5_Utilizo técnicas de NLP (Natural Language Processing) para análisar dados não-estruturados.": "Utilizo_tecnicas_de_NLP_Natural_Language_Processing_para_analisar_dados_nao_estruturados",
    "8.b.6_Utilizo métodos estatísticos clássicos (Testes de hipótese, análise multivariada, sobrevivência, dados longitudinais, inferência estatistica) para analisar dados.": "Utilizo_metodos_estatisticos_classicos_Testes_de_hipotese_analise_multivariada_sobrevivencia_dados_longitudinais_inferencia_estatistica_para_analisar_dados",
    "8.b.7_Utilizo cadeias de Markov ou HMM\\s para realizar análises de dados.": "Utilizo_cadeias_de_Markov_ou_HMM_s_para_realizar_analises_de_dados",
    "8.b.8_Desenvolvo técnicas de Clusterização (K-means, Spectral, DBScan etc).": "Desenvolvo_tecnicas_de_Clusterizacao_K_means_Spectral_DBScan_etc",
    "8.b.9_Realizo previsões através de modelos de Séries Temporais (Time Series).": "Realizo_previsoes_atraves_de_modelos_de_Series_Temporais_Time_Series",
    "8.b.10_Utilizo modelos de Reinforcement Learning (aprendizado por reforço).": "Utilizo_modelos_de_Reinforcement_Learning_aprendizado_por_reforco",
    "8.b.11_Utilizo modelos de Machine Learning para detecção de fraude.": "Utilizo_modelos_de_Machine_Learning_para_deteccao_de_fraude",
    "8.b.12_Utilizo métodos de Visão Computacional.": "Utilizo_metodos_de_Visao_Computacional",
    "8.b.13_Utilizo modelos de Detecção de Churn.": "Utilizo_modelos_de_Deteccao_de_Churn",
    "8.b.14_Utilizo LLM's para solucionar problemas de negócio.": "Utilizo_LLM_s_para_solucionar_problemas_de_negocio",
    "8.c_tecnologias_ds": "Quais_dessas_tecnologias_fazem_parte_do_seu_dia_a_dia_como_cientista_de_dados",
    "8.c.1_Ferramentas de BI (PowerBI, Looker, Tableau, Qlik etc).": "Ferramentas_de_BI_PowerBI_Looker_Tableau_Qlik_etc",
    "8.c.2_Planilhas (Excel, Google Sheets etc).": "Planilhas_Excel_Google_Sheets_etc",
    "8.c.3_Ambientes de desenvolvimento local (R-studio, JupyterLab, Anaconda).": "Ambientes_de_desenvolvimento_local_R_studio_JupyterLab_Anaconda",
    "8.c.4_Ambientes de desenvolvimento na nuvem (Google Colab, AWS Sagemaker, Kaggle Notebooks etc).": "Ambientes_de_desenvolvimento_na_nuvem_Google_Colab_AWS_Sagemaker_Kaggle_Notebooks_etc",
    "8.c.5_Ferramentas de AutoML (Datarobot, H2O, Auto-Keras etc).": "Ferramentas_de_AutoML_Datarobot_H2O_Auto_Keras_etc",
    "8.c.6_Ferramentas de ETL (Apache Airflow, NiFi, Stitch, Fivetran, Pentaho etc).": "Ferramentas_de_ETL_Apache_Airflow_NiFi_Stitch_Fivetran_Pentaho_etc",
    "8.c.7_Plataformas de Machine Learning (TensorFlow, Azure Machine Learning, Kubeflow etc).": "Plataformas_de_Machine_Learning_TensorFlow_Azure_Machine_Learning_Kubeflow_etc",
    "8.c.8_Feature Store (Feast, Hopsworks, AWS Feature Store, Databricks Feature Store etc).": "Feature_Store_Feast_Hopsworks_AWS_Feature_Store_Databricks_Feature_Store_etc",
    "8.c.9_Sistemas de controle de versão (Github, DVC, Neptune, Gitlab etc).": "Sistemas_de_controle_de_versao_Github_DVC_Neptune_Gitlab_etc",
    "8.c.10_Plataformas de Data Apps (Streamlit, Shiny, Plotly Dash etc).": "Plataformas_de_Data_Apps_Streamlit_Shiny_Plotly_Dash_etc",
    "8.c.11_Ferramentas de estatística avançada como SPSS, SAS, etc.": "Ferramentas_de_estatistica_avancada_como_SPSS_SAS_etc",
    "8.d_maior_tempo_gasto_como_ds": "Em_qual_das_opcoes_abaixo_voce_gasta_a_maior_parte_do_seu_tempo_no_trabalho",
    "8.d.2_Coletando e limpando dos dados que uso para análise e modelagem.": "Coletando_e_limpando_os_dados_que_uso_para_analise_e_modelagem",
    "8.d.3_Entrando em contato com os times de negócio para definição do problema, identificar a solução e apresentação de resultados.": "Entrando_em_contato_com_os_times_de_negocio_para_definicao_do_problema_identificar_a_solucao_e_apresentacao_de_resultados",
    "8.d.4_Desenvolvendo modelos de Machine Learning com o objetivo de colocar em produção em sistemas (produtos de dados).": "Desenvolvendo_modelos_de_Machine_Learning_com_o_objetivo_de_colocar_em_producao_em_sistemas_produtos_de_dados",
    "8.d.5_Colocando modelos em produção, criando os pipelines de dados, APIs de consumo e monitoramento.": "Colocando_modelos_em_producao_criando_os_pipelines_de_dados_APIs_de_consumo_e_monitoramento",
    "8.d.6_Cuidando da manutenção de modelos de Machine Learning já em produção, atuando no monitoramento, ajustes e refatoração quando necessário.": "Cuidando_da_manutencao_de_modelos_de_Machine_Learning_ja_em_producao_atuando_no_monitoramento_ajustes_e_refatoracao_quando_necessario",
    "8.d.9_Criando e dando manutenção em ETLs, DAGs e automações de pipelines de dados.": "Criando_e_dando_manutencao_em_ETLs_DAGs_e_automacoes_de_pipelines_de_dados",
    "8.d.10_Criando e gerenciando soluções de Feature Store e cultura de MLOps.": "Criando_e_gerenciando_solucoes_de_Feature_Store_e_cultura_de_MLOps",
    "8.d.11_Criando e mantendo a infra que meus modelos e soluções rodam (clusters, servidores, API, containers, etc.)": "Criando_e_mantendo_a_infra_que_meus_modelos_e_solucoes_rodam_clusters_servidores_API_containers_etc",
    "8.d.12_Treinando e aplicando LLM's para solucionar problemas de negócio.": "Treinando_e_aplicando_LLM_s_para_solucionar_problemas_de_negocio",
}

## 6. Aplicar o mapeamento e corrigir o tipo das colunas

In [ ]:
# Aplica o mapeamento no df_2023
df_2023_padronizado = df_2023
for coluna in df_2023.columns:
    if coluna in mapa_2023:
        novo_nome = mapa_2023[coluna]
        df_2023_padronizado = (
            df_2023_padronizado
            .withColumnRenamed(coluna, novo_nome)
            .withColumn(novo_nome, col(novo_nome).cast("string"))
        )
        
print("Colunas após padronização: ", len(df_2023_padronizado.columns))
print(df_2023_padronizado.columns[:15])

In [ ]:
# Aplica o mapeamento no df_2024
df_2024_padronizado = df_2024
for coluna in df_2024.columns:
    if coluna in mapa_2024:
        novo_nome = mapa_2024[coluna]
        df_2024_padronizado = (
            df_2024_padronizado
            .withColumnRenamed(coluna, novo_nome)
            .withColumn(novo_nome, col(novo_nome).cast("string"))
        )

print("Colunas após padronização: ", len(df_2024_padronizado.columns))
print(df_2024_padronizado.columns[:15])

In [ ]:
# Aplica o mapeamento no df_2025
df_2025_padronizado = df_2025
for coluna in df_2025.columns:
    if coluna in mapa_2025:
        novo_nome = mapa_2025[coluna]
        df_2025_padronizado = (
            df_2025_padronizado
            .withColumnRenamed(coluna, novo_nome)
            .withColumn(novo_nome, col(novo_nome).cast("string"))
        )

print("Colunas após padronização: ", len(df_2025_padronizado.columns))
print(df_2025_padronizado.columns[:15])

## 7. Unificar os 3 anos

In [ ]:
df_unificado = (
    df_2023_padronizado
    .unionByName(df_2024_padronizado, allowMissingColumns=True)
    .unionByName(df_2025_padronizado, allowMissingColumns=True)
)

print("Total de linhas unificadas:", df_unificado.count())
df_unificado.select(df_unificado.columns[:15]).show(5)

## 8. Gravar a base unificada em Parquet

In [33]:
df_unificado.write.mode("overwrite").option("compression", "snappy").parquet(CAMINHO_SILVER_UNIFICADO)

print(f"Base unificada gravada em {CAMINHO_SILVER_UNIFICADO}")

Base unificada gravada em s3://tech-challenge-fase-3-grupo-94/silver/state_of_data_unificado/
